# Spacetime Gaussian Feature Splatting (oppo-us-research/SpacetimeGaussians): Neural 3D Video multi-view benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riccardolanza05/dynamic-gaussian-splatting-benchmark/blob/main/notebooks/07_spacetime_gaussians_li_n3dv.ipynb)

Part of **dynamic-gaussian-splatting-benchmark**, extended to the **multi-view** setting: a controlled comparison of three dynamic Gaussian Splatting methods on the six scenes of the *Neural 3D Video Synthesis from Multi-View Video* (N3DV) dataset, run on a cloud GPU (lightning.ai Studio, Google Colab, or any Linux machine with an NVIDIA card). The method is trained with its official training code; a benchmark monitor attached to its training loop records PSNR, SSIM, LPIPS, evaluation L1, training time, iteration count, number of Gaussians, peak VRAM and model storage into one JSON file per run — the same monitor, the same fields and the same two protocols as the monocular D-NeRF study in notebooks `01`–`03`.

**AI disclosure.** This notebook, the rest of the repository and part of the code used to extract the metrics (the benchmark monitor, the per-scene preparation, the dataset converters and the analysis scripts) were generated with AI assistance (Anthropic's Claude), starting from the code of the official repository of each method. The training code of the method itself is the official one, cloned at run time; the only changes are the monitor hook appended to `train.py`, configuration overrides and environment fixes, all made by the cells below and listed in `docs/METHODOLOGY_MULTIVIEW.md`.

**Method.** Spacetime Gaussian Feature Splatting for Real-Time Dynamic View Synthesis, Li et al., CVPR 2024 ([paper](https://arxiv.org/abs/2312.16812), [code](https://github.com/oppo-us-research/SpacetimeGaussians)). Gaussians with a temporal radial-basis opacity and polynomial motion and rotation, initialised from a COLMAP model per frame; a sequence is covered in chunks of 50 frames.

This method was **studied but not benchmarked** in the monocular part of the project: it needs synchronised multi-camera video and cannot run on D-NeRF. This is its first notebook, written to the same template as `01`–`03`. **Both released variants are benchmarked**, as two rows of the results: `ours_lite`, the direct-RGB model and the default of this notebook, and `ours_full`, which adds a small MLP colour decoder that neither of the other two methods has. `STG_MODEL` switches between the two; with `ours_full` the decoder is saved next to the point cloud (`point_cloud.pt`) and counted in the model storage.

Two protocols. Protocol B is unchanged from the monocular study; Protocol A no longer forces one step count on every method:

* **Protocol A, official budget**: every method trains for the number of optimisation steps its authors use on N3DV (4DGaussians 3 000 + 14 000, 4DGS native-4D 30 000, Spacetime Gaussians 30 000), with the metrics sampled about 30 times along the run.
* **Protocol B, equal quality**: training stops when the evaluation L1 reaches a per-scene target shared by the three notebooks, metrics sampled every 30 seconds.

**Multi-view conventions** (identical in notebooks `05`–`07`): N3DV at **1352×1014** (half of the native 2704×2028); **all 300 frames** of every scene in the main study, as the papers report, and the **first 50 frames** in the short-window study (`NUM_FRAMES = 50`); **`cam00` held out** as the test camera, so the test split is one view per frame of the window; LPIPS with the VGG backbone from a single backend; evaluation L1 as the common loss. Why each of these was needed is in [`docs/METHODOLOGY_MULTIVIEW.md`](https://github.com/riccardolanza05/dynamic-gaussian-splatting-benchmark/blob/main/docs/METHODOLOGY_MULTIVIEW.md).

**Notebook structure**

*Part 1: configuration, setup and training*

0. User configuration and path resolution
1. Environment and repository setup
2. CUDA module and environment fixes
3. N3DV dataset download and per-scene preparation (per-frame COLMAP stage)
4. Training with incremental benchmarking (single scene or loop, Protocol B calibration, summary)

*Part 2: evaluation and visualisation*

5. Rendering and final metrics
6. Video export of the held-out view

---

# Part 1: configuration, setup and training

This part is safe to run unattended. It configures the run, prepares the environment and the dataset, installs the benchmark instrumentation and trains one scene or a list of scenes. It produces model checkpoints and one JSON summary per run.

**Do not run Part 2 during a training loop.** It renders and computes metrics, which compete for the GPU and would corrupt the training-time and peak-VRAM measurements.

---

## 0. Platform, configuration and paths

Cell 0.0 works out **where this notebook is running** and nothing else. The three
environments it supports need different base directories and different ways of reaching
Google Drive:

| | base directory | Google Drive |
|---|---|---|
| **lightning.ai Studio** | `/teamspace/studios/this_studio` (persistent) | `rclone` remote |
| **Google Colab** | `/content` (lost when the session ends) | `drive.mount()` |
| **any other Linux machine** | `~/dgs-benchmark` | `rclone` remote |

Detection is automatic; `BENCH_PLATFORM` and `BENCH_WORKDIR` override it if needed.

Cell 0.0 also defines the `env_*` helpers that make **every setting in cell 0.1
overridable from an environment variable**, so the notebook can be driven from a terminal
over SSH without editing a single cell. Each setting keeps its literal as the default, so
with no variables set the notebook behaves exactly as if you had edited cell 0.1 by hand.
How to drive a run from a terminal: see the section *Running on a cloud GPU over SSH* of the repository [README](https://github.com/riccardolanza05/dynamic-gaussian-splatting-benchmark#running-on-a-cloud-gpu-over-ssh).

In [ ]:
# 0.0  Where are we running, and how are settings overridden?
import os
import shutil
import subprocess
import sys


def detect_platform():
    """"colab", "lightning" or "generic". Never raises."""
    if "google.colab" in sys.modules:
        return "colab"
    try:
        import google.colab  # noqa: F401
        return "colab"
    except Exception:
        pass
    # A lightning.ai Studio always has this persistent directory.
    if os.path.isdir("/teamspace/studios/this_studio"):
        return "lightning"
    return "generic"


PLATFORM = os.environ.get("BENCH_PLATFORM") or detect_platform()

DEFAULT_WORKDIR = {
    "colab": "/content",
    "lightning": "/teamspace/studios/this_studio",
}.get(PLATFORM, os.path.expanduser("~/dgs-benchmark"))

# Everything this notebook creates -- cloned repositories, the dataset cache, the models,
# the results -- lives under WORKDIR. On a Studio it is persistent; on Colab it is not.
WORKDIR = os.environ.get("BENCH_WORKDIR") or DEFAULT_WORKDIR
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)


# Settings from the environment: every value in cell 0.1 can be overridden with a
# BENCH_<NAME> variable, which is what makes an unattended SSH run possible. With none
# set, the literals written in cell 0.1 are used.

def env_str(name, default):
    value = os.environ.get("BENCH_" + name)
    return default if value is None or value == "" else value


def env_int(name, default):
    value = os.environ.get("BENCH_" + name)
    return default if value is None or value == "" else int(value)


def env_float(name, default):
    value = os.environ.get("BENCH_" + name)
    return default if value is None or value == "" else float(value)


def env_bool(name, default):
    value = os.environ.get("BENCH_" + name)
    if value is None or value == "":
        return default
    return value.strip().lower() in ("1", "true", "yes", "on")


def env_list(name, default):
    """A comma-separated list, e.g. BENCH_SCENES="sear_steak,flame_steak"."""
    value = os.environ.get("BENCH_" + name)
    if value is None or value == "":
        return list(default)
    return [item.strip() for item in value.split(",") if item.strip()]


# Installing system packages needs root on a Studio but not on Colab.
if os.geteuid() == 0:
    SUDO = ""
elif shutil.which("sudo"):
    SUDO = "sudo -n "
else:
    SUDO = ""

print("Platform    : %s" % PLATFORM)
print("Working dir : %s" % WORKDIR)
_overrides = sorted(k for k in os.environ if k.startswith("BENCH_"))
print("Overrides   : %s" % (", ".join(_overrides) if _overrides else "none (using the "
                            "literals in cell 0.1)"))
try:
    print()
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv"], capture_output=True, text=True).stdout.strip()
          or "nvidia-smi produced no output: is a GPU attached?")
except FileNotFoundError:
    print("nvidia-smi not found: this machine has no NVIDIA driver. Training will fail.")


## 0. User configuration

Everything that changes between runs is set in cell 0.1; no other cell needs editing.

**Storage.** `STORAGE_MODE = "drive"` writes the benchmark JSON (and, with `KEEP_MODEL_ON_DRIVE = True`, the model) to Google Drive, so a run survives a runtime restart and the loop can resume. `"local"` keeps everything under the working directory of the machine (`/content` on Colab, where it is lost when the session ends).

**What to run.** `RUN_MODE = "single"` trains `SCENE`; `"loop"` trains every scene in `SCENES_TO_RUN`. Completed runs are skipped, so re-running a loop after a disconnection resumes from the first missing scene.

**Which protocol.**

| | Protocol A, `"iterations"` | Protocol B, `"target_eval_loss"` |
|---|---|---|
| Stopping criterion | the method's official N3DV budget, `MAX_ITERATIONS` | per-scene target in `TARGET_EVAL_LOSS_PER_SCENE` |
| Metric sampling | every `EVAL_EVERY_N_ITERS` iterations, about 30 samples per run | every `EVAL_EVERY_N_MINUTES` minutes |
| Question answered | run as its authors run it, how well does each method reconstruct, and at what cost? | to reach the same quality, what does each method cost? |

**Workflow.** The Protocol B target must be reachable by every method, so it is calibrated from Protocol A: run the Protocol A loop on all scenes in all three notebooks, read the candidates printed by cell 4.3, take the per-scene maximum across the methods and use the same dictionary in all three notebooks. **The targets ship as `None`**, because no multi-view run exists yet; with `None` the Protocol B loop skips the scene instead of inventing a bar. A set of targets belongs to **one window**: targets calibrated on 50-frame runs must not be used for a 300-frame run.

**The temporal window.** `NUM_FRAMES = 300` is the whole sequence (10 s at 30 fps) and the main study: it is what the three papers report, so the results can be set next to the published ones. `NUM_FRAMES = 50` (frames 0–49, 1.67 s) is the short-window study, which is where Protocol B is run. Spacetime Gaussians trains N3DV in 50-frame blocks by construction, so its 300-frame run is six independent models, merged into one result (notebook `07`). The window is used for training *and* for the test split, and it is written into every benchmark JSON (`num_frames`) and into every run folder name, so runs made with different windows can never be mixed by accident.

**Disk.** One N3DV scene is about 1.2 GB of video. The extracted frames are estimated at 2–3 MB per PNG at 1352×1014, i.e. roughly 2–3 GB for ~20 cameras × 50 frames and 13–18 GB for 300 frames; this is an estimate, to be replaced by the figure the first run records (`scene_data_mb` in the benchmark JSON). `DELETE_VIDEOS_AFTER_EXTRACT` and `DELETE_FRAMES_AFTER_TRAIN` keep the machine inside its disk budget when looping over several scenes.

In [ ]:
# 0.1  User configuration. Edit the defaults here, or set the matching
# BENCH_<NAME> environment variable to override one without touching the file.

# Where the results go.
#   "drive" : the shared Google Drive folder of the study. On Colab that folder is mounted
#             directly; anywhere else it is a local staging directory kept in step with
#             Drive by rclone (cell 0.2), so an SSH run needs no browser.
#   "local" : stay on this machine, sync nothing.
STORAGE_MODE = env_str("STORAGE_MODE", "drive")

# ONE folder for the whole multi-view study: the three notebooks write three subfolders
# inside it, named exactly as results/analysis/scripts/study.py expects, so the folder can
# be downloaded and analysed as it is. This is what collects every JSON of every method.
BENCH_ROOT_NAME = env_str("ROOT_NAME", "dgs-benchmark-n3dv")
METHOD_DIR_NAME = "spacetime_gaussians_output"
# Name of the rclone remote pointing at Google Drive (created once with `rclone config`).
RCLONE_REMOTE = env_str("RCLONE_REMOTE", "gdrive")

# "single" trains SCENE only; "loop" trains every scene in SCENES_TO_RUN, in order.
RUN_MODE = env_str("RUN_MODE", "loop")

SCENE = env_str("SCENE", "sear_steak")

# Used only when RUN_MODE == "loop"; trim the list for a partial loop.
SCENES_TO_RUN = env_list("SCENES", ["coffee_martini", "cook_spinach", "cut_roasted_beef",
                                    "flame_salmon_1", "flame_steak", "sear_steak"])
SCENES_NOT_TO_RUN = []

# Skip runs that are already complete (this makes the loop resumable).
SKIP_TRAINING_IF_TRAINED = env_bool("SKIP_TRAINING_IF_TRAINED", True)

# False: the model stays on local disk and only the benchmark JSON is kept
#        (model storage is measured before the local copy is discarded).
# True:  the model is kept next to the JSON, so Part 2 can run later without retraining.
#        On Drive that is gigabytes per run: leave it False unless you need it.
KEEP_MODEL_ON_DRIVE = env_bool("KEEP_MODEL_ON_DRIVE", False)
LOCAL_TRAIN_ROOT = os.path.join(WORKDIR, "train_outputs")

# --- Multi-view protocol (identical in notebooks 05-07) -----------------------------
# Frames 0..NUM_FRAMES-1 of every scene are used for training and for the test split.
# 300 = the whole sequence, the main study (what the papers report); 50 = the short-window
# study, where Protocol B is run.
NUM_FRAMES = env_int("NUM_FRAMES", 300)
# N3DV holds out the centre camera; the dataset README states cam00 is the test view.
HELD_OUT_CAMERA = env_str("HELD_OUT_CAMERA", "cam00")
# Evaluation resolution: half of the native 2704x2028, the resolution all three official
# configs use for N3DV. Written as a downscale factor of the native video.
IMAGE_DOWNSCALE = env_int("IMAGE_DOWNSCALE", 2)
# ------------------------------------------------------------------------------------

# Where the raw N3DV videos are cached. One scene is about 1.2 GB.
N3DV_RAW_ROOT = env_str("N3DV_RAW_ROOT", os.path.join(WORKDIR, "n3dv"))
# Delete the .mp4 files once the frames have been extracted (they are not needed again).
DELETE_VIDEOS_AFTER_EXTRACT = env_bool("DELETE_VIDEOS_AFTER_EXTRACT", True)
# Delete the extracted frames of a scene once its run is finished (gigabytes per scene).
DELETE_FRAMES_AFTER_TRAIN = env_bool("DELETE_FRAMES_AFTER_TRAIN", True)

# Benchmarking protocol: "iterations" (Protocol A) or "target_eval_loss" (Protocol B).
TRAINING_MODE = env_str("TRAINING_MODE", "iterations")

# Protocol A: the budget this method's authors use on N3DV, not a number shared by the
# three methods (docs/OPEN_DECISIONS_MULTIVIEW.md, decision 6).
# Official: 30 000 steps (the default of arguments/__init__.py). The N3DV configs
# then test one snapshot per scene (test_iteration: 25 000 on four scenes, 10 000 on
# coffee_martini, 12 000 on flame_salmon_1), recorded as official_test_iteration.
MAX_ITERATIONS     = env_int("MAX_ITERATIONS", 30000)
# Sampling grid, chosen per method so that a run gives about 30 samples of its curve.
EVAL_EVERY_N_ITERS = env_int("EVAL_EVERY_N_ITERS", 1000)

# Protocol B: fixed quality target
# "eval_loss" stops on the per-scene L1 target below, "psnr" on TARGET_PSNR.
TARGET_METRIC = "eval_loss"
TARGET_PSNR   = 30.0    # used only when TARGET_METRIC == "psnr"

# Per-scene L1 target, identical in the three notebooks, to be calibrated from the Protocol A
# results of all three methods (cell 4.3, docs/PROTOCOL_B_CALIBRATION.md), on the same
# window the Protocol B runs will use.
# A scene left at None is skipped by the Protocol B loop: no multi-view Protocol A run exists
# yet, so every target is None and must be filled in before Protocol B can be run.
TARGET_EVAL_LOSS_PER_SCENE = {
    "coffee_martini":   None,
    "cook_spinach":     None,
    "cut_roasted_beef": None,
    "flame_salmon_1":   None,
    "flame_steak":      None,
    "sear_steak":       None,
}

EVAL_EVERY_N_MINUTES  = 0.5     # sample the metrics every T minutes
SAFETY_MAX_ITERATIONS = 60000   # hard cap, so a run can never last forever
# The target is ignored below this iteration, so the baseline sample cannot stop the run.
MIN_ITERATIONS_BEFORE_STOP = 1000
# The target must hold for this many consecutive samples (test metrics fluctuate).
TARGET_CONSECUTIVE_HITS = 2

# Evaluation settings (shared by both protocols)
# "l1": mean L1 on the test split; "l1_dssim": (1-w)*L1 + w*(1-SSIM). Both are always recorded;
# this selects the one mirrored by "eval_loss" and used by the Protocol B stop.
EVAL_LOSS_KIND = "l1"
LAMBDA_DSSIM   = 0.2      # weight w of the "l1_dssim" variant

LPIPS_NET      = "vgg"    # LPIPS backbone, as reported in all three papers
# One LPIPS backend for the whole multi-view study: the pip package (with the [-1,1]
# rescaling it expects) in all three notebooks, whatever each repository bundles, so that
# the LPIPS column cannot carry an offset between two implementations.
LPIPS_BACKEND  = "lpips-pip"
MAX_EVAL_VIEWS = 0        # 0 = every test view (the held-out camera, one per frame)
EVAL_AT_FIRST_ITERATION = True   # record a baseline point at the first iteration

# Extra flags appended to the training command (usually empty)
EXTRA_TRAIN_ARGS = env_str("EXTRA_TRAIN_ARGS", "")

# Views consumed per optimiser step; the official per-scene config may overwrite it.
BATCH_SIZE = 2

# Which of the two released models this run trains. Both are benchmarked, as two rows of
# the results: "ours_lite" is the direct-RGB model and the default here, "ours_full" adds an
# MLP colour decoder that no other method in this comparison has (see
# docs/OPEN_DECISIONS_MULTIVIEW.md, decision 3). Run the notebook once per variant.
STG_MODEL = env_str("STG_MODEL", "ours_lite")
assert STG_MODEL in ("ours_lite", "ours_full")
# ours_full runs get their own folders (<scene>_f300_iters30000_full), so switching the
# variant never reuses or overwrites a run of the other one.
RUN_TAG = "" if STG_MODEL == "ours_lite" else "_full"

# The official unit of this method is a block of 50 frames (duration: 50 in every N3DV
# config). Up to 50 frames the window is one block. Beyond it -- the 300-frame main study,
# docs/OPEN_DECISIONS_MULTIVIEW.md, decision 2 -- the scene is trained as its
# paper does: NUM_FRAMES / 50 independent models, one per block, whose results are merged
# (quality averaged over all test views, time and storage summed, VRAM the maximum).
STG_BLOCK_FRAMES = 50
if NUM_FRAMES <= STG_BLOCK_FRAMES:
    NUM_BLOCKS = 1
else:
    assert NUM_FRAMES % STG_BLOCK_FRAMES == 0, (
        "Beyond %d frames the window must be a multiple of %d (one model per block)"
        % (STG_BLOCK_FRAMES, STG_BLOCK_FRAMES))
    NUM_BLOCKS = NUM_FRAMES // STG_BLOCK_FRAMES
print("Blocks   : %d x %d frames" % (NUM_BLOCKS, min(NUM_FRAMES, STG_BLOCK_FRAMES)))

METHOD_NAME = "Spacetime Gaussian Feature Splatting (Li et al.)"
REPO_DIR = os.path.join(WORKDIR, "SpacetimeGaussians")
N3DV_SCENES = ["coffee_martini", "cook_spinach", "cut_roasted_beef",
               "flame_salmon_1", "flame_steak", "sear_steak"]

assert STORAGE_MODE in ("drive", "local")
assert RUN_MODE in ("single", "loop")
assert TRAINING_MODE in ("iterations", "target_eval_loss")
assert EVAL_LOSS_KIND in ("l1", "l1_dssim")
assert TARGET_METRIC in ("psnr", "eval_loss")
assert LPIPS_BACKEND in ("lpips-pip", "lpipsPyTorch", "auto")
assert STORAGE_MODE in ("drive", "local")
assert 1 <= NUM_FRAMES <= 300, "N3DV provides 300 frames per camera"
assert SCENE in N3DV_SCENES, "Unknown scene: %s" % SCENE
assert all(s in N3DV_SCENES for s in SCENES_TO_RUN)
print("Configuration accepted.")
print("Method   : %s" % METHOD_NAME)
print("Run mode : %s" % RUN_MODE)
print("Protocol : %s" % TRAINING_MODE)
print("Window   : frames 0..%d, test camera %s" % (NUM_FRAMES - 1, HELD_OUT_CAMERA))
print("Scenes   : %s" % (SCENE if RUN_MODE == "single" else ", ".join(SCENES_TO_RUN)))

In [ ]:
# 0.2  Where the results live, and how they reach Google Drive.
import glob
import json
import os
import shutil
import subprocess

# STUDY_ROOT is the one folder that holds all three methods; OUTPUT_ROOT is this method's
# subfolder of it. The layout is the one the analysis pipeline reads directly:
#
#   <STUDY_ROOT>/
#       4dgaussian_n3dv_output/      <run>/benchmark/benchmark_*.json      (notebook 05)
#       4dgs_fudan_n3dv_output/      ...                                   (notebook 06)
#       spacetime_gaussians_output/  ...                                   (notebook 07)
#
DRIVE_SYNC = None          # rclone destination, or None when nothing is synced

if STORAGE_MODE == "drive" and PLATFORM == "colab":
    # Colab can mount Drive as a filesystem: write straight into it.
    from google.colab import drive
    drive.mount("/content/drive")
    STUDY_ROOT = os.path.join("/content/drive/MyDrive", BENCH_ROOT_NAME)
elif STORAGE_MODE == "drive":
    # Everywhere else (a lightning.ai Studio, any Linux box): write locally and keep the
    # folder in step with Drive through rclone, which needs no browser once configured.
    STUDY_ROOT = os.path.join(WORKDIR, "bench_out", BENCH_ROOT_NAME)
    DRIVE_SYNC = "%s:%s" % (RCLONE_REMOTE, BENCH_ROOT_NAME)
else:
    STUDY_ROOT = os.path.join(WORKDIR, "bench_out", BENCH_ROOT_NAME)

OUTPUT_ROOT = os.path.join(STUDY_ROOT, METHOD_DIR_NAME)

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(LOCAL_TRAIN_ROOT, exist_ok=True)
os.makedirs(N3DV_RAW_ROOT, exist_ok=True)
TRAINED_GLOB = ("point_cloud", "iteration_*", "point_cloud.ply")


def _rclone(*args):
    """Run rclone, returning True on success. Never raises: a failed sync must not
    destroy a finished training run, it must only be reported."""
    if shutil.which("rclone") is None:
        print("[sync] rclone is not installed; results stay in %s only." % OUTPUT_ROOT)
        return False
    command = ["rclone"] + list(args)
    print("[sync] $ %s" % " ".join(command), flush=True)
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        print("[sync] FAILED (%d): %s" % (result.returncode,
                                          (result.stderr or "").strip()[-400:]))
        return False
    return True


def sync_pull():
    """Fetch this method's results back from Drive, so a fresh machine can resume.

    Only the benchmark folders are pulled, never the models: what makes the loop
    resumable is the JSON, and the models can be gigabytes.
    """
    if not DRIVE_SYNC:
        return
    # */blocks/** holds the per-block results of a run trained in blocks (notebook 07 at
    # more than 50 frames), so a half-finished scene resumes at the first missing block.
    _rclone("copy", "%s/%s" % (DRIVE_SYNC, METHOD_DIR_NAME), OUTPUT_ROOT,
            "--include", "*/benchmark/**", "--include", "*/blocks/**",
            "--create-empty-src-dirs")


def sync_push(run=None):
    """Send results to Drive. With a run name, only that run; otherwise everything."""
    if not DRIVE_SYNC:
        return
    if run:
        source = os.path.join(OUTPUT_ROOT, run)
        destination = "%s/%s/%s" % (DRIVE_SYNC, METHOD_DIR_NAME, run)
    else:
        source, destination = OUTPUT_ROOT, "%s/%s" % (DRIVE_SYNC, METHOD_DIR_NAME)
    if not os.path.isdir(source):
        return
    # copy, not sync: nothing on Drive is ever deleted by a run.
    if _rclone("copy", source, destination, "--transfers", "4"):
        print("[sync] %s -> %s" % (source, destination))


def run_name(scene, mode):
    """Folder name for one (scene, protocol) pair. Runs never collide.

    The frame window is part of the name: multi-view runs made with different
    windows are different experiments and must not share a folder.
    """
    # RUN_TAG names a model variant the rest of the name does not capture (notebook 07 sets
    # it for ours_full), so two variants never share a folder or reuse each other's JSON.
    suffix = globals().get("RUN_TAG", "")
    if mode == "iterations":
        return "%s_f%d_iters%d%s" % (scene, NUM_FRAMES, MAX_ITERATIONS, suffix)
    target = TARGET_EVAL_LOSS_PER_SCENE.get(scene)
    tag = ("%g" % target).replace(".", "p") if target is not None else "NA"
    return "%s_f%d_loss%s%s" % (scene, NUM_FRAMES, tag, suffix)


def raw_scene_dir(scene):
    """Where the raw N3DV videos and poses_bounds.npy of one scene live."""
    return os.path.join(N3DV_RAW_ROOT, scene)


def source_path_for(scene):
    """The COLMAP model of the first frame of the window; the reader walks to colmap_1, colmap_2 ... from there, one model per frame."""
    return os.path.join(REPO_DIR, "data", "n3dv", scene, "colmap_0")


def paths_for(scene, mode):
    """All output paths for one run.

    The benchmark JSON always lives on OUTPUT_ROOT (Drive when selected), while
    the model goes to Drive only if KEEP_MODEL_ON_DRIVE, otherwise to local
    disk. Metrics are what the benchmark needs; the checkpoints are optional.
    """
    name = run_name(scene, mode)
    model_root = OUTPUT_ROOT if KEEP_MODEL_ON_DRIVE else LOCAL_TRAIN_ROOT
    model_path = os.path.join(model_root, name)
    bench_dir = os.path.join(OUTPUT_ROOT, name, "benchmark")
    return {
        "scene": scene,
        "mode": mode,
        "run_name": name,
        "model_path": model_path,
        "drive_model_path": os.path.join(OUTPUT_ROOT, name),
        "benchmark_dir": bench_dir,
        "bench_json": os.path.join(bench_dir, "benchmark_%s.json" % mode),
        "bench_config": os.path.join(bench_dir, "benchmark_config.json"),
        "source_path": source_path_for(scene),
        "raw_path": raw_scene_dir(scene),
    }


def is_model_trained(model_path):
    """True when the model files of that run are present on disk."""
    return len(glob.glob(os.path.join(model_path, *TRAINED_GLOB))) > 0


TERMINAL_STATUS = ("finished", "stopped_on_target", "stopped_on_safety_cap")


def run_is_complete(p):
    """True when this run has already produced a finished benchmark JSON.

    Completion is judged on the JSON rather than on the model, because with
    KEEP_MODEL_ON_DRIVE = False the model does not survive the runtime. This is
    what makes the loop resumable after a disconnection or a GPU-quota stop.
    """
    if not os.path.exists(p["bench_json"]):
        return False
    try:
        with open(p["bench_json"], "r") as handle:
            summary = json.load(handle)
    except Exception:
        return False
    if summary.get("status") not in TERMINAL_STATUS:
        return False
    if summary.get("num_frames") not in (None, NUM_FRAMES):
        return False
    if KEEP_MODEL_ON_DRIVE and not is_model_trained(p["drive_model_path"]):
        return False
    return True


def activate_scene(scene, mode=None):
    """Point the module-level variables at one scene. Part 2 reads these."""
    global SCENE, SOURCE_PATH, MODEL_OUTPUT_PATH, MODEL_DIR, RAW_PATH
    global BENCHMARK_DIR, BENCH_JSON_PATH, BENCH_CONFIG_PATH, RUN_NAME
    mode = mode or TRAINING_MODE
    p = paths_for(scene, mode)
    SCENE = scene
    SOURCE_PATH = p["source_path"]
    RAW_PATH = p["raw_path"]
    MODEL_OUTPUT_PATH = p["model_path"]
    MODEL_DIR = p["model_path"]          # alias used by Part 2
    BENCHMARK_DIR = p["benchmark_dir"]
    BENCH_JSON_PATH = p["bench_json"]
    BENCH_CONFIG_PATH = p["bench_config"]
    RUN_NAME = run_name(scene, mode)
    os.makedirs(BENCHMARK_DIR, exist_ok=True)
    return p


activate_scene(SCENE)

# A fresh Studio starts empty: bring back what previous runs already produced, so the
# loop skips them instead of retraining them.
sync_pull()

print("Study root   : %s" % STUDY_ROOT)
print("Metrics root : %s" % OUTPUT_ROOT)
print("Drive sync   : %s" % (DRIVE_SYNC or "off (results stay on this machine)"))
print("Model folder : %s%s" % (MODEL_OUTPUT_PATH,
                               "" if KEEP_MODEL_ON_DRIVE else "   (not kept)"))
print("Benchmark    : %s" % BENCH_JSON_PATH)
print("Already done : %s" % run_is_complete(paths_for(SCENE, TRAINING_MODE)))

## 1. Environment and repository setup

In [ ]:
# The GPU this run will use (cell 0.0 already printed a summary)
!nvidia-smi

In [ ]:
import os
os.chdir(WORKDIR)
!git clone --recursive https://github.com/oppo-us-research/SpacetimeGaussians
%cd SpacetimeGaussians
!git submodule update --init --recursive

# Reuse the PyTorch/CUDA already installed and add the remaining dependencies
!pip install -q natsort plyfile tqdm scipy configargparse tensorboard opencv-python-headless lpips "imageio[ffmpeg]" matplotlib

In [ ]:
# The CUDA modules of this repository live under thirdparty/gaussian_splatting/submodules.
# Only the rasterizers the benchmarked model needs are built: ch3 for ours_lite,
# ch9 for ours_full. The forward-only ones are used by the real-time viewer, not here.
import os

!pip install -q "setuptools<70" wheel ninja
%env TORCH_CUDA_ARCH_LIST=7.5

os.chdir(os.path.join(REPO_DIR, 'thirdparty/gaussian_splatting/submodules/simple-knn'))
!pip install . --no-build-isolation

_module = ('gaussian_rasterization_ch3' if STG_MODEL == 'ours_lite'
           else 'gaussian_rasterization_ch9')
os.chdir(os.path.join(REPO_DIR, 'thirdparty/gaussian_splatting/submodules', _module))
!pip install . --no-build-isolation

os.chdir(REPO_DIR)
print('CUDA modules installed for %s.' % STG_MODEL)

## 2. CUDA module and environment fixes

* `simple-knn` needs `<float.h>` for its floating-point limits, as in every repository of this comparison.
* The rasterizers are reinstalled without `-e` and with `--no-build-isolation`, against a `setuptools` version compatible with PyTorch's CUDA extensions.
* `thirdparty/gaussian_splatting` is not a package on `sys.path` by default; `train.py` appends it at import time, and the benchmark monitor does the same before importing the metric modules.

No reader code is patched in this notebook: the temporal window is `--duration`, a command-line value.

In [ ]:
import os

# simple-knn needs <float.h>
knn_dir = os.path.join(REPO_DIR, "thirdparty/gaussian_splatting/submodules/simple-knn")
filepath = os.path.join(knn_dir, "simple_knn.cu")
with open(filepath, "r") as f:
    content = f.read()

if "#include <float.h>" not in content:
    with open(filepath, "w") as f:
        f.write("#include <float.h>\n" + content)
    print("Patch applied; re-run the installation cell above.")
else:
    print("Patch already present.")

os.chdir(REPO_DIR)

In [ ]:
import subprocess

for mod in ("natsort", "plyfile", "lpips", "matplotlib", "imageio", "cv2",
            "simple_knn", "torch"):
    r = subprocess.run("python -c 'import %s'" % mod, shell=True,
                       capture_output=True, text=True)
    print("  %-34s %s" % (mod, "OK" if r.returncode == 0 else "MISSING"))

_raster = ("diff_gaussian_rasterization_ch3" if STG_MODEL == "ours_lite"
           else "diff_gaussian_rasterization_ch9")
r = subprocess.run("python -c 'import %s'" % _raster, shell=True,
                   capture_output=True, text=True)
print("  %-34s %s" % (_raster, "OK" if r.returncode == 0 else "MISSING"))

r = subprocess.run(
    "cd %s && python -c \"import sys; sys.path.append('./thirdparty/gaussian_splatting'); "
    "from thirdparty.gaussian_splatting.scene import Scene; print('OK')\"" % REPO_DIR,
    shell=True, capture_output=True, text=True)
print("\n  repo importable by train.py        %s"
      % (r.stdout.strip() or r.stderr.strip()[-300:]))

## 3. N3DV dataset download

The *Neural 3D Video Synthesis from Multi-View Video* dataset is downloaded **one scene at a time**, because a single scene is about 1.2 GB of video plus gigabytes of extracted frames (an estimated 13–18 GB at 300 frames), well beyond what a single machine can comfortably hold for six scenes at once. The download is cached under `N3DV_RAW_ROOT` and skipped when the scene is already there, so the cell is safe to re-run.

Each scene ships as `cam00.mp4 … cam20.mp4` plus `poses_bounds.npy` in the LLFF pose format. `cam00` is the centre reference camera and is **held out for testing** — that is the dataset's own convention, stated in its README, and the convention all three methods follow. `flame_salmon_1` is published as a split archive and is rejoined before extraction.

Frames are extracted at **1352×1014** (`IMAGE_DOWNSCALE = 2`), the resolution used by the official N3DV configuration of every method in this comparison.

In [ ]:
# 3  N3DV download and frame extraction, one scene at a time (resumable).
import os
import shutil
import subprocess

N3DV_RELEASE = ("https://github.com/facebookresearch/Neural_3D_Video/"
                "releases/download/v1.0")
# flame_salmon_1 is published as a 4-part split zip; the others as a single archive.
N3DV_SPLIT_SCENES = {"flame_salmon_1": ["flame_salmon_1_split.z01",
                                        "flame_salmon_1_split.z02",
                                        "flame_salmon_1_split.z03",
                                        "flame_salmon_1_split.zip"]}


def _sh(cmd, cwd=None):
    print("$ %s" % cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    assert proc.returncode == 0, "command failed (%d): %s" % (proc.returncode, cmd)


def scene_is_downloaded(scene):
    d = raw_scene_dir(scene)
    if not os.path.exists(os.path.join(d, "poses_bounds.npy")):
        return False
    return len([f for f in os.listdir(d) if f.endswith(".mp4")]) > 0


def scene_is_extracted(scene, num_frames=None):
    """True when every camera already has the frames of the window on disk."""
    num_frames = NUM_FRAMES if num_frames is None else num_frames
    d = raw_scene_dir(scene)
    cams = sorted(c for c in os.listdir(d)
                  if os.path.isdir(os.path.join(d, c)) and c.startswith("cam"))
    if not cams:
        return False
    for cam in cams:
        frames = os.path.join(d, cam, "frames")
        if not os.path.isdir(frames):
            return False
        if any(not os.path.exists(os.path.join(frames, "%04d.png" % i))
               for i in range(num_frames)):
            return False
    return True


def download_scene(scene):
    """Fetch one N3DV scene into raw_scene_dir(scene). Skipped when present."""
    target = raw_scene_dir(scene)
    if scene_is_downloaded(scene):
        print("[%s] videos already present, download skipped." % scene)
        return target
    os.makedirs(target, exist_ok=True)
    tmp = os.path.join(N3DV_RAW_ROOT, "_dl_%s" % scene)
    os.makedirs(tmp, exist_ok=True)
    if scene in N3DV_SPLIT_SCENES:
        for part in N3DV_SPLIT_SCENES[scene]:
            dst = os.path.join(tmp, part)
            if not os.path.exists(dst):
                _sh('wget -q --show-progress -O "%s" "%s/%s"' % (dst, N3DV_RELEASE, part))
        joined = os.path.join(tmp, "joined.zip")
        if not os.path.exists(joined):
            _sh('zip -q -s 0 "%s/%s" --out "%s"'
                % (tmp, N3DV_SPLIT_SCENES[scene][-1], joined))
        _sh('unzip -q -o "%s" -d "%s"' % (joined, tmp))
    else:
        archive = os.path.join(tmp, "%s.zip" % scene)
        if not os.path.exists(archive):
            _sh('wget -q --show-progress -O "%s" "%s/%s.zip"'
                % (archive, N3DV_RELEASE, scene))
        _sh('unzip -q -o "%s" -d "%s"' % (archive, tmp))

    # The archives unpack either flat or inside a <scene>/ folder; normalise both.
    src = tmp
    if not os.path.exists(os.path.join(src, "poses_bounds.npy")):
        candidates = [os.path.join(src, d) for d in os.listdir(src)
                      if os.path.isdir(os.path.join(src, d))
                      and os.path.exists(os.path.join(src, d, "poses_bounds.npy"))]
        assert candidates, "poses_bounds.npy not found in the archive of %s" % scene
        src = candidates[0]
    for name in os.listdir(src):
        if name.endswith(".mp4") or name == "poses_bounds.npy":
            shutil.move(os.path.join(src, name), os.path.join(target, name))
    shutil.rmtree(tmp, ignore_errors=True)
    print("[%s] downloaded to %s" % (scene, target))
    return target


def extract_frames(scene):
    """Decode the first NUM_FRAMES frames of every camera at 1352x1014.

    Frames land in <raw>/<camNN>/frames/%04d.png, which is the layout every
    per-method converter below reads. Cameras keep the names of their videos, so
    the held-out camera stays identifiable whatever the pose ordering is.
    """
    d = raw_scene_dir(scene)
    videos = sorted(f for f in os.listdir(d) if f.endswith(".mp4"))
    assert videos, "no videos in %s; run download_scene first" % d
    for video in videos:
        cam = os.path.splitext(video)[0]
        out = os.path.join(d, cam, "frames")
        os.makedirs(out, exist_ok=True)
        done = all(os.path.exists(os.path.join(out, "%04d.png" % i))
                   for i in range(NUM_FRAMES))
        if done:
            continue
        scale = ("-vf scale=iw/%d:ih/%d" % (IMAGE_DOWNSCALE, IMAGE_DOWNSCALE)
                 if IMAGE_DOWNSCALE != 1 else "")
        _sh('ffmpeg -y -loglevel error -i "%s" -vframes %d %s -start_number 0 "%s/%%04d.png"'
            % (os.path.join(d, video), NUM_FRAMES, scale, out))
    if DELETE_VIDEOS_AFTER_EXTRACT:
        for video in videos:
            os.remove(os.path.join(d, video))
        print("[%s] videos removed after extraction (poses_bounds.npy kept)." % scene)
    print("[%s] %d frames per camera at downscale %d."
          % (scene, NUM_FRAMES, IMAGE_DOWNSCALE))


def ensure_scene_raw(scene):
    """Download and extract one scene if needed; returns its raw folder."""
    if not scene_is_extracted(scene):
        if not scene_is_downloaded(scene):
            download_scene(scene)
        extract_frames(scene)
    else:
        print("[%s] frames already extracted, nothing to do." % scene)
    return raw_scene_dir(scene)


def drop_scene_frames(scene):
    """Free the extracted frames of a scene once its run is finished."""
    d = raw_scene_dir(scene)
    for cam in sorted(os.listdir(d)):
        frames = os.path.join(d, cam, "frames")
        if os.path.isdir(frames):
            shutil.rmtree(frames, ignore_errors=True)
    print("[%s] extracted frames discarded." % scene)


def n3dv_cameras(scene):
    """Camera names of a scene, in the order of poses_bounds.npy.

    The dataset README warns that unsynchronised streams were removed, so the
    numbering has gaps and poses_bounds.npy follows the *sorted file order*, not
    the camera index. Sorting the names reproduces that order exactly.
    """
    d = raw_scene_dir(scene)
    cams = sorted(c for c in os.listdir(d)
                  if os.path.isdir(os.path.join(d, c)) and c.startswith("cam"))
    if not cams:
        cams = sorted(os.path.splitext(f)[0] for f in os.listdir(d)
                      if f.endswith(".mp4"))
    return cams


print("Dataset helpers ready. Raw root: %s" % N3DV_RAW_ROOT)
print("Scenes already extracted: %s"
      % [s for s in N3DV_SCENES
         if os.path.isdir(raw_scene_dir(s)) and scene_is_extracted(s)])

## 3.4 Per-frame COLMAP stage

This method initialises from **one COLMAP model per frame** — not one for the whole sequence — because its primitives carry a time centre and a polynomial trajectory, and the per-frame point clouds are what gives them a temporal spread. That is intrinsic to the method, so notebook `07` does not share the single-frame stage of notebooks `05`–`06`: it runs the reconstruction `NUM_FRAMES` times, as `script/pre_n3d.py` does.

The stage is the upstream one, called function by function, with one substitution: the frames come from cell 3 (already extracted at 1352×1014) instead of being decoded again at full resolution by `extractframes`. `convertdynerftocolmapdb` and `getcolmapsinglen3d` — the pose conversion and the COLMAP calls — are the repository's own, imported rather than copied.

This is the slowest preparation of the three notebooks: one COLMAP run on ~20 images per frame of the window (300 in the main study, 50 in the short-window one). It is cached per scene and resumable, so an interrupted runtime does not restart it.

**More than 50 frames.** The official unit of this method is a block of 50 frames. With `NUM_FRAMES` above 50 (a multiple of 50: the 300-frame main study, `docs/OPEN_DECISIONS_MULTIVIEW.md`, decision 2) cell 4 trains the scene **as the paper does, one independent model per block**: block *k* covers frames 50k–50k+49 and is read from `colmap_<50k>` with `--duration 50`. Each block is an ordinary upstream run with the monitor attached, and its JSON is kept under `<run>/blocks/block_<k>/`. When every block has reached its budget the results are merged into the run's single benchmark JSON: quality (PSNR, SSIM, LPIPS, L1) is the mean over all test views of all blocks, training time, images seen, Gaussians and storage are summed, and peak VRAM is the maximum. `iteration` stays the per-block step, and `total_iterations` is the sum over the blocks. An interrupted scene resumes at the first unfinished block. Protocol B is not run in blocks, because its "first time the target is reached" is not defined for a set of independent models; how it could be defined is an open decision (`docs/OPEN_DECISIONS_MULTIVIEW.md`, decision 7).

In [ ]:
# 3.4  Per-frame COLMAP stage (upstream pre_n3d.py, with our already-extracted frames).
import glob
import json
import os
import shutil
import sys

import numpy as np

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)


def install_colmap():
    if shutil.which("colmap"):
        print("colmap already installed: %s" % shutil.which("colmap"))
        return
    # SUDO is "" on Colab (already root) and "sudo -n " on a Studio.
    _sh(SUDO + "apt-get -qq update")
    _sh(SUDO + "DEBIAN_FRONTEND=noninteractive apt-get -qq install -y colmap")
    print("colmap installed: %s" % shutil.which("colmap"))


def stg_scene_dir(scene):
    """The folder this method reads: poses_bounds.npy, cam??/<i>.png and colmap_<i>/."""
    return os.path.join(REPO_DIR, "data", "n3dv", scene)


def link_frames(scene):
    """Lay the extracted frames out as <scene>/cam??/<i>.png, as pre_n3d.py does.

    Upstream writes one PNG per frame with an unpadded index; the frames extracted
    in cell 3 are zero-padded, so they are linked, not copied.
    """
    raw = raw_scene_dir(scene)
    dst_root = stg_scene_dir(scene)
    os.makedirs(dst_root, exist_ok=True)
    shutil.copyfile(os.path.join(raw, "poses_bounds.npy"),
                    os.path.join(dst_root, "poses_bounds.npy"))
    for cam in n3dv_cameras(scene):
        # convertdynerftocolmapdb() globs cam*.mp4 to recover the camera order, which must
        # match poses_bounds.npy; an empty placeholder is enough, no video is ever opened.
        placeholder = os.path.join(dst_root, "%s.mp4" % cam)
        if not os.path.exists(placeholder):
            open(placeholder, "a").close()
        out = os.path.join(dst_root, cam)
        os.makedirs(out, exist_ok=True)
        for i in range(NUM_FRAMES):
            dst = os.path.join(out, "%d.png" % i)
            if not os.path.exists(dst):
                os.symlink(os.path.abspath(os.path.join(raw, cam, "frames", "%04d.png" % i)),
                           dst)
    return dst_root


def run_per_frame_colmap(scene):
    """Run the upstream per-frame COLMAP preparation; resumable, cached per scene."""
    from pathlib import Path

    from script.pre_n3d import preparecolmapdynerf, convertdynerftocolmapdb
    from thirdparty.gaussian_splatting.helper3dg import getcolmapsinglen3d

    install_colmap()
    root = Path(link_frames(scene))
    for offset in range(NUM_FRAMES):
        done = os.path.join(str(root), "colmap_%d" % offset, "sparse", "0", "points3D.bin")
        if os.path.exists(done):
            continue
        preparecolmapdynerf(root, offset)
        # The frames are already downscaled, so the intrinsics are divided by the same factor.
        convertdynerftocolmapdb(root, offset, IMAGE_DOWNSCALE)
        getcolmapsinglen3d(str(root), offset)
        print("[%s] COLMAP model %d/%d ready." % (scene, offset + 1, NUM_FRAMES), flush=True)
    return str(root)


def prepare_scene(scene, mode):
    """Prepare the data; this method is driven entirely by the command line."""
    ensure_scene_raw(scene)
    run_per_frame_colmap(scene)
    return EXTRA_TRAIN_ARGS


CONFIG_DIR = {"ours_lite": "n3d_lite", "ours_full": "n3d_full"}
# SpacetimeGaussians names flame_salmon's config after the dataset folder.
OFFICIAL_CONFIG = {s: s for s in N3DV_SCENES}


def config_path(scene):
    return os.path.join(REPO_DIR, "configs", CONFIG_DIR[STG_MODEL],
                        "%s.json" % OFFICIAL_CONFIG[scene])


def build_train_command(p, budget, extra):
    # --resolution 1: the frames were extracted at 1352x1014 already, so the official
    #   "resolution": 2 (which halves 2704x2028) must not be applied a second time.
    # --duration is the length of the block being trained: the study's window up to 50
    #   frames, 50 for each block beyond (six blocks at the default NUM_FRAMES = 300).
    cmd = ('python train.py'
           + ' --quiet --eval'
           + ' --configpath "%s"' % config_path(p["scene"])
           + ' --source_path "%s"' % p["source_path"]
           + ' --model_path "%s"' % p["model_path"]
           + ' --model %s' % STG_MODEL
           + ' --rdpip train_%s' % STG_MODEL
           + ' --duration %d' % p.get("num_frames", NUM_FRAMES)
           + ' --resolution 1'
           + ' --iterations %d' % budget
           + ' --test_iterations -1'
           + ' --save_iterations %d' % budget)
    return cmd + ((' ' + extra) if extra else '')


def extra_train_env(p, budget):
    return {}


def extra_bench_config(p):
    import json as _json

    with open(config_path(p["scene"])) as handle:
        official = _json.load(handle)
    blocks = {"stg_block_frames": STG_BLOCK_FRAMES, "stg_num_blocks": NUM_BLOCKS}
    if "block_index" in p:
        blocks.update({"block_index": p["block_index"], "block_start_frame": p["block_start"],
                       "study_num_frames": int(NUM_FRAMES)})
    return {"init_point_cloud": "sparse_per_frame",
            "stg_model": STG_MODEL,
            **blocks,
            "official_config": os.path.relpath(config_path(p["scene"]), REPO_DIR),
            "official_duration": official.get("duration"),
            # The snapshot upstream's test.py evaluates on this scene: not the same on every
            # scene (25 000 on four, 10 000 on coffee_martini, 12 000 on flame_salmon_1).
            "official_test_iteration": official.get("test_iteration"),
            "densify": official.get("densify")}


def scene_data_dirs(scene):
    """Extra folders this method fills for a scene (the per-frame COLMAP models)."""
    return [stg_scene_dir(scene)]


def release_scene(scene):
    shutil.rmtree(stg_scene_dir(scene), ignore_errors=True)
    drop_scene_frames(scene)


def _folder_size_mb(root):
    total = 0
    for dirpath, _d, filenames in os.walk(root):
        for fn in filenames:
            try:
                total += os.path.getsize(os.path.join(dirpath, fn))
            except OSError:
                pass
    return total / 1048576.0


def _sum_patterns(model_path, patterns):
    files = []
    for pat in patterns:
        files += sorted(glob.glob(os.path.join(model_path, pat)))
    files = [f for f in files if os.path.isfile(f)]
    total = sum(os.path.getsize(f) for f in files)
    return files, total


# The model of ours_lite is the point cloud alone: save_ply() writes every per-Gaussian
# attribute (position, time centre and scale, motion and omega polynomials, colour, opacity,
# scale, rotation) into point_cloud.ply. cfg_args is what the rendering scripts read back.
# For ours_full the MLP decoder is also needed: oursfull.py's save_ply() writes it to
# point_cloud.pt next to the .ply (and load_ply() reads it back), so it is counted here.
REQUIRED_PATTERNS = ("cfg_args",
                     "point_cloud/iteration_*/point_cloud.ply",
                     "point_cloud/iteration_*/point_cloud.pt")


def model_storage_report(model_path):
    files, total = _sum_patterns(model_path, REQUIRED_PATTERNS)
    return {
        "required_files": [os.path.relpath(f, model_path) for f in files],
        "model_storage_mb": round(total / 1048576.0, 4),
        "full_folder_mb": round(_folder_size_mb(model_path), 4),
        "stg_model": STG_MODEL,
    }


# --- Training in blocks (NUM_FRAMES > 50: the 300-frame main study) --------------------
# Block k covers frames 50k .. 50k+49. Upstream reads it from colmap_<50k> with
# --duration 50: the reader walks colmap_<50k> .. colmap_<50k+49>, holds out cam00 for
# those frames and rescales time to the block. Each block is an ordinary run of the
# upstream train.py with the monitor attached; its JSON goes to <run>/blocks/block_<k>/,
# outside <run>/benchmark/, so the analysis only ever sees the merged result.

def block_paths(p, k):
    start = k * STG_BLOCK_FRAMES
    block_dir = os.path.join(p["drive_model_path"], "blocks", "block_%d" % k)
    q = dict(p)
    q.update({
        "block_index": k,
        "block_start": start,
        "num_frames": STG_BLOCK_FRAMES,
        "source_path": os.path.join(stg_scene_dir(p["scene"]), "colmap_%d" % start),
        "model_path": os.path.join(p["model_path"], "block_%d" % k),
        "benchmark_dir": block_dir,
        "bench_json": os.path.join(block_dir, "benchmark_%s.json" % p["mode"]),
        "bench_config": os.path.join(block_dir, "benchmark_config.json"),
    })
    return q


def _read_json(path):
    try:
        with open(path, "r") as handle:
            return json.load(handle)
    except Exception:
        return None


def block_is_complete(q, budget):
    """A block counts as done only if its run reached the full budget: the monitor marks
    an interrupted process 'finished' too, so the status alone is not enough."""
    summary = _read_json(q["bench_json"])
    if not summary or summary.get("status") not in TERMINAL_STATUS:
        return False
    final = summary.get("final") or {}
    cfg = summary.get("config") or {}
    return (final.get("iteration", -1) >= budget
            and cfg.get("block_start_frame") == q["block_start"]
            and cfg.get("stg_model") == STG_MODEL
            and summary.get("process_returncode", 0) == 0)


def merge_block_summaries(p, blocks):
    """One benchmark JSON for the whole window, in the schema of every other run.

    Definitions, those of the paper where it has one:
      * quality (PSNR, SSIM, LPIPS, L1): mean over all test views of all blocks, i.e.
        the per-block means weighted by their number of views (equal blocks: plain mean);
      * training time, wall time, overhead, images seen, steps, Gaussians: summed;
      * storage: summed; peak VRAM: the maximum over the blocks.
    Entries are merged at the iterations every block sampled (the same grid in Protocol A);
    'iteration' stays the per-block step, 'total_iterations' is the sum over the blocks.
    """
    grids = [{e["iteration"]: e for e in b["entries"]} for b in blocks]
    common = sorted(set.intersection(*[set(g) for g in grids]))
    entries = []
    for it in common:
        es = [g[it] for g in grids]
        views = sum(e["num_eval_views"] for e in es)

        def wmean(key):
            return sum(e[key] * e["num_eval_views"] for e in es) / float(views)

        losses = [e["train_batch_loss"] for e in es if e.get("train_batch_loss") is not None]
        entries.append({
            "iteration": it,
            "total_iterations": sum(e["total_iterations"] for e in es),
            "images_seen": sum(e["images_seen"] for e in es),
            "stage": "single_stage",
            "training_time_s": sum(e["training_time_s"] for e in es),
            "wall_time_s": sum(e["wall_time_s"] for e in es),
            "benchmark_overhead_s": sum(e["benchmark_overhead_s"] for e in es),
            "psnr": wmean("psnr"),
            "ssim": wmean("ssim"),
            "lpips": wmean("lpips"),
            "eval_l1_loss": wmean("eval_l1_loss"),
            "eval_photometric_loss": wmean("eval_photometric_loss"),
            "eval_loss": wmean("eval_loss"),
            "eval_loss_kind": es[0]["eval_loss_kind"],
            "num_eval_views": views,
            "eval_resolution": es[0]["eval_resolution"],
            "num_gaussians": sum(e["num_gaussians"] for e in es),
            "train_batch_loss": (sum(losses) / len(losses)) if losses else None,
            "eval_duration_s": sum(e.get("eval_duration_s", 0.0) for e in es),
            "blocks": len(es),
        })

    storages = [b.get("storage") or {} for b in blocks]
    summary = {
        "method": METHOD_NAME,
        "scene": p["scene"],
        "mode": p["mode"],
        "dataset": "n3dv",
        "num_frames": int(NUM_FRAMES),
        "held_out_camera": HELD_OUT_CAMERA,
        "config": _read_json(p["bench_config"]) or {},
        "status": "finished",
        "target_reached": False,
        "entries": entries,
        "final": entries[-1] if entries else None,
        "lpips_backend": blocks[0].get("lpips_backend"),
        "process_wall_time_s": sum(b.get("process_wall_time_s") or 0.0 for b in blocks),
        "peak_vram_mb_nvidia_smi": max(b.get("peak_vram_mb_nvidia_smi") or 0.0 for b in blocks),
        "storage": {
            "model_storage_mb": round(sum(s.get("model_storage_mb") or 0.0 for s in storages), 4),
            "full_folder_mb": round(sum(s.get("full_folder_mb") or 0.0 for s in storages), 4),
            "required_files": ["block_%d/%s" % (k, f) for k, s in enumerate(storages)
                               for f in s.get("required_files", [])],
            "per_block_mb": [s.get("model_storage_mb") for s in storages],
            "stg_model": STG_MODEL,
        },
        "blocks": [{"index": k, "start_frame": k * STG_BLOCK_FRAMES,
                    "frames": STG_BLOCK_FRAMES,
                    "final_psnr": (b.get("final") or {}).get("psnr"),
                    "json": os.path.relpath(block_paths(p, k)["bench_json"],
                                            p["drive_model_path"])}
                   for k, b in enumerate(blocks)],
    }
    if entries:
        best = max(entries, key=lambda e: e["psnr"])
        summary["best_psnr"] = best["psnr"]
        summary["best_psnr_iteration"] = best["iteration"]
    return summary


def train_scene_in_blocks(scene, mode):
    """Train one scene as NUM_BLOCKS independent 50-frame models, then merge them."""
    if mode != "iterations":
        # Protocol B has no defined meaning across six separate models: "the first time
        # the target is reached" does not exist for a set of independent runs.
        print("[%s] Protocol B is not defined for a run in %d blocks -> skipped."
              % (scene, NUM_BLOCKS))
        return "skipped_protocol_b_in_blocks"
    p = activate_scene(scene, mode)
    budget = MAX_ITERATIONS
    merged = _read_json(p["bench_json"])
    if (SKIP_TRAINING_IF_TRAINED and merged and merged.get("status") in TERMINAL_STATUS
            and merged.get("num_frames") == NUM_FRAMES):
        print("[%s] already benchmarked -> skipped (%s)." % (scene, p["bench_json"]))
        return "skipped_already_trained"

    os.chdir(REPO_DIR)
    _t_prep = time.time()
    extra = prepare_scene(scene, mode)       # per-frame COLMAP for the whole window
    prep_s = time.time() - _t_prep
    data_mb = scene_data_mb(scene)
    write_bench_config(p)                    # the run's own config, kept in the merged JSON
    statuses = []
    for k in range(NUM_BLOCKS):
        q = block_paths(p, k)
        if SKIP_TRAINING_IF_TRAINED and block_is_complete(q, budget):
            print("[%s] block %d/%d already trained -> skipped." % (scene, k + 1, NUM_BLOCKS))
            statuses.append("ok")
            continue
        print("\n[%s] block %d/%d: frames %d-%d"
              % (scene, k + 1, NUM_BLOCKS, q["block_start"],
                 q["block_start"] + STG_BLOCK_FRAMES - 1), flush=True)
        write_bench_config(q)
        cmd = build_train_command(q, budget, extra)
        returncode, wall, peak = run_training_process(q, cmd, budget)
        summary = _read_json(q["bench_json"]) or {}
        if summary:
            summary["process_returncode"] = returncode
            with open(q["bench_json"], "w") as handle:
                json.dump(summary, handle, indent=2)
        record_storage(q)
        if not KEEP_MODEL_ON_DRIVE:
            shutil.rmtree(q["model_path"], ignore_errors=True)
        try:
            sync_push(p["run_name"])           # each finished block reaches Drive at once
        except Exception as exc:
            print("[%s] sync failed: %r" % (scene, exc))
        statuses.append("ok" if returncode == 0 else "failed(%d)" % returncode)
        print("[%s] block %d/%d finished in %dm %ds (return code %d), peak VRAM %.0f MB"
              % (scene, k + 1, NUM_BLOCKS, int(wall // 60), int(wall % 60),
                 returncode, peak), flush=True)

    blocks = [_read_json(block_paths(p, k)["bench_json"]) for k in range(NUM_BLOCKS)]
    if not all(block_is_complete(block_paths(p, k), budget) for k in range(NUM_BLOCKS)):
        print("[%s] not every block completed (%s): no merged result written; re-run to "
              "resume from the missing blocks." % (scene, statuses), flush=True)
        return "incomplete_blocks"

    merged = merge_block_summaries(p, blocks)
    # Of this invocation: near zero when the per-frame COLMAP models were already cached.
    merged["preparation_time_s"] = prep_s
    merged["scene_data_mb"] = data_mb
    os.makedirs(p["benchmark_dir"], exist_ok=True)
    with open(p["bench_json"], "w") as handle:
        json.dump(merged, handle, indent=2)
    final = merged["final"] or {}
    print("[%s] %d blocks merged: PSNR %.3f over %d test views, training %.1f min, "
          "storage %.1f MB, peak VRAM %.0f MB"
          % (scene, NUM_BLOCKS, final.get("psnr", float("nan")),
             final.get("num_eval_views", 0), final.get("training_time_s", 0.0) / 60.0,
             merged["storage"]["model_storage_mb"], merged["peak_vram_mb_nvidia_smi"]),
          flush=True)
    if not KEEP_MODEL_ON_DRIVE:
        shutil.rmtree(p["model_path"], ignore_errors=True)
    try:
        sync_push(p["run_name"])
    except Exception as exc:
        print("[%s] sync failed: %r" % (scene, exc))
    if DELETE_FRAMES_AFTER_TRAIN:
        try:
            release_scene(scene)
        except Exception as exc:
            print("[%s] could not free the scene data: %r" % (scene, exc))
    return "ok"


print("Benchmarked variant : %s" % STG_MODEL)
print("Config directory    : configs/%s" % CONFIG_DIR[STG_MODEL])
print("Per-frame COLMAP    : %d models per scene" % NUM_FRAMES)
_available = sorted(os.path.basename(f)[:-5] for f in
                    glob.glob(os.path.join(REPO_DIR, "configs", CONFIG_DIR[STG_MODEL], "*.json")))
print("Official configs    : %s" % _available)

## 4. Training with incremental benchmarking

**Instrumentation.** The training loop of each repository calls one of its own functions at every iteration. Instead of rewriting the loop, which would alter densification schedules, learning-rate ramps and stage handling, the notebook appends a few lines to `train.py` that wrap that function with a monitor. The upstream code path, hyper-parameters and command line stay untouched, a pristine copy is kept as `train.py.orig`, and without the `BENCH_CONFIG` environment variable `train.py` behaves exactly as upstream.

Two of the three repositories expose a `training_report(...)` that already receives the loop state, as in the monocular study. The third (Spacetime Gaussians) does not have such a function at all: its loop calls a narrower per-iteration function, so the monitor recovers the loop state from the **calling frame** instead of from the signature. The frame is only read, never written, and the wrapped function is still an upstream function the loop already called at every iteration.

**The monitor core is byte-identical in notebooks `05`–`07`** — it is delimited by the `BENCHMARK MONITOR CORE` markers — and everything that differs between repositories lives in the glue appended after it. At every sampling point the monitor evaluates the current model on the **full test split** (`cam00`, every frame of the window: 300 views in the main study, 50 in the short-window one) with each repository's own metric modules and records PSNR, SSIM, LPIPS (VGG), the evaluation L1, the training time net of the benchmark overhead, the iteration count, the images seen, the number of Gaussians and the evaluation resolution. The JSON summary is rewritten atomically after each evaluation, so an interrupted run still leaves a valid file.

**Protocol B cost.** A run stops only after the target has held for `TARGET_CONSECUTIVE_HITS` consecutive samples, so the termination time overstates the cost, by a different amount for each method. The figure to report is the **first crossing**, extracted by cell 4.4.

In [ ]:
# 4.0  Install the benchmark monitor and build the training driver
import json
import os
import shutil
import subprocess
import sys
import threading
import time

os.chdir(REPO_DIR)

MONITOR_SRC = r'''# benchmark_monitor.py -- generated by notebook 07: incremental benchmark logger for
# Spacetime Gaussian Feature Splatting (Li et al.) on Neural 3D Video.
# This repository has no training_report(...): its loop calls controlgaussians(...) at every
# iteration, so that is the function the notebook wraps, and the loop state that the narrower
# signature does not carry (the render pipeline, the pipeline parameters, the background and
# the current loss) is read from the calling frame.

import sys as _sys

_sys.path.append("./thirdparty/gaussian_splatting")

# --- repository metric modules: the only import block that differs between notebooks 05-07 ---
from thirdparty.gaussian_splatting.utils.image_utils import psnr as _repo_psnr
from thirdparty.gaussian_splatting.utils.loss_utils import ssim as _repo_ssim
from thirdparty.gaussian_splatting.utils.loss_utils import l1_loss as _repo_l1

# === BENCHMARK MONITOR CORE (byte-identical in notebooks 05-07) ===
# Everything between this marker and END BENCHMARK MONITOR CORE is shared verbatim by the
# three multi-view notebooks. The repository-specific glue is appended after it and provides
# _stage_is_evaluable, _global_iteration, _num_gaussians, _train_batch_loss, _force_eval,
# _iter_eval_views, _render_pair, _save_model and _maybe_track_best.

import atexit
import inspect
import json
import os
import sys
import time

import torch


# Configuration written by the notebook, passed through $BENCH_CONFIG

_CFG_PATH = os.environ.get("BENCH_CONFIG", "")

with open(_CFG_PATH, "r") as _f:
    CFG = json.load(_f)

METHOD = CFG["method"]
SCENE = CFG["scene"]
MODE = CFG["mode"]                                   # "iterations" | "target_eval_loss"
JSON_PATH = CFG["json_path"]

EVAL_EVERY_N_ITERS = int(CFG.get("eval_every_n_iters", 2000))
EVAL_EVERY_N_SECONDS = float(CFG.get("eval_every_n_seconds", 600.0))
MAX_ITERATIONS = int(CFG.get("max_iterations", 30000))
SAFETY_MAX_ITERATIONS = int(CFG.get("safety_max_iterations", 100000))

# Protocol B stopping criterion: "eval_loss" (eval_loss <= target) or "psnr" (psnr >= target)
TARGET_METRIC = CFG.get("target_metric", "psnr")
TARGET_PSNR = float(CFG.get("target_psnr", 30.0))
TARGET_EVAL_LOSS = float(CFG.get("target_eval_loss", 0.0))
MIN_ITERATIONS_BEFORE_STOP = int(CFG.get("min_iterations_before_stop", 2000))
# Test metrics are not monotonic: the target must hold for several consecutive samples.
TARGET_CONSECUTIVE_HITS = int(CFG.get("target_consecutive_hits", 2))

# Evaluation loss: a common photometric yardstick, not any method's own objective
# (regularisers never look at the ground truth and have no evaluation counterpart).
EVAL_LOSS_KIND = CFG.get("eval_loss_kind", "l1")     # "l1" | "l1_dssim"
LAMBDA_DSSIM = float(CFG.get("lambda_dssim", 0.2))
LPIPS_NET = CFG.get("lpips_net", "vgg")
# "lpips-pip" | "lpipsPyTorch" | "auto". The multi-view study forces one backend for all its
# methods, whatever each repository bundles, because a mixed backend would put a systematic
# offset on the LPIPS column.
LPIPS_BACKEND = CFG.get("lpips_backend", "auto")
MAX_EVAL_VIEWS = int(CFG.get("max_eval_views", 0))   # 0 = all test views
EVAL_AT_FIRST_ITERATION = bool(CFG.get("eval_at_first_iteration", True))

# BATCH_SIZE: views consumed per optimiser step (per-method, from the official config).
# ITERATION_OFFSET: steps of an earlier stage whose counter restarted (coarse stage of Wu et al.).
BATCH_SIZE = int(CFG.get("batch_size", 1))
ITERATION_OFFSET = int(CFG.get("iteration_offset", 0))

# Multi-view protocol: the temporal window and the held-out camera, recorded in every entry so
# that a JSON file is self-describing and two runs can never be compared across windows.
NUM_FRAMES = int(CFG.get("num_frames", 0))
HELD_OUT_CAMERA = CFG.get("held_out_camera", "cam00")
DATASET = CFG.get("dataset", "n3dv")


# LPIPS: one backend for the whole study; the model is built once.

_lpips_model = None
_lpips_backend = None


def _build_lpips_pip():
    import lpips as _lpips_pkg
    return _lpips_pkg.LPIPS(net=LPIPS_NET).to("cuda").eval(), "lpips-pip"


def _build_lpips_bundled():
    from lpipsPyTorch.modules.lpips import LPIPS
    return LPIPS(net_type=LPIPS_NET).to("cuda").eval(), "lpipsPyTorch"


def _get_lpips():
    global _lpips_model, _lpips_backend
    if _lpips_model is not None:
        return _lpips_model, _lpips_backend
    if LPIPS_BACKEND == "lpips-pip":
        _lpips_model, _lpips_backend = _build_lpips_pip()
    elif LPIPS_BACKEND == "lpipsPyTorch":
        _lpips_model, _lpips_backend = _build_lpips_bundled()
    else:
        try:
            _lpips_model, _lpips_backend = _build_lpips_bundled()
        except Exception as exc:
            print("[benchmark] lpipsPyTorch unavailable (%r), falling back to the "
                  "lpips pip package." % (exc,))
            _lpips_model, _lpips_backend = _build_lpips_pip()
    _SUMMARY["lpips_backend"] = _lpips_backend
    print("[benchmark] LPIPS backend: %s (%s)" % (_lpips_backend, LPIPS_NET), flush=True)
    return _lpips_model, _lpips_backend


def _lpips_value(image, gt):
    model, backend = _get_lpips()
    x = image.unsqueeze(0)
    y = gt.unsqueeze(0)
    if backend == "lpips-pip":
        # the pip package expects inputs in [-1, 1]
        x = x * 2.0 - 1.0
        y = y * 2.0 - 1.0
    return model(x, y).mean().item()


# JSON summary, rewritten atomically at every evaluation

_SUMMARY = {
    "method": METHOD,
    "scene": SCENE,
    "mode": MODE,
    "dataset": DATASET,
    "num_frames": NUM_FRAMES,
    "held_out_camera": HELD_OUT_CAMERA,
    "config": CFG,
    "status": "running",
    "target_reached": False,
    "entries": [],
    "final": None,
}


def _flush_summary():
    directory = os.path.dirname(JSON_PATH)
    if directory:
        os.makedirs(directory, exist_ok=True)
    tmp_path = JSON_PATH + ".tmp"
    with open(tmp_path, "w") as handle:
        json.dump(_SUMMARY, handle, indent=2)
    os.replace(tmp_path, JSON_PATH)


@atexit.register
def _on_exit():
    if _SUMMARY["status"] == "running":
        _SUMMARY["status"] = "finished"
    if _SUMMARY["entries"]:
        _SUMMARY["final"] = _SUMMARY["entries"][-1]
    try:
        _flush_summary()
    except Exception as exc:
        print("[benchmark] could not write the JSON summary: %r" % (exc,))


class _Monitor(object):

    def __init__(self):
        self.t_start = None
        self.eval_overhead_s = 0.0
        self.last_eval_time = None
        self.last_eval_iteration = -1
        self.best_psnr = -1.0
        self.n_evals = 0
        self.consecutive_hits = 0

    def _target_met(self, entry):
        if entry["iteration"] < MIN_ITERATIONS_BEFORE_STOP:
            return False
        if TARGET_METRIC == "psnr":
            return entry["psnr"] >= TARGET_PSNR
        return entry["eval_loss"] <= TARGET_EVAL_LOSS

    def _must_evaluate(self, a, iteration, now):
        if iteration == self.last_eval_iteration:
            return False
        if EVAL_AT_FIRST_ITERATION and self.n_evals == 0:
            return True
        if _force_eval(a):
            return True
        if MODE == "iterations":
            if iteration >= MAX_ITERATIONS:
                return True
            return (iteration % EVAL_EVERY_N_ITERS) == 0
        # MODE == "target_eval_loss": time-driven sampling
        if iteration >= SAFETY_MAX_ITERATIONS:
            return True
        return (now - self.last_eval_time) >= EVAL_EVERY_N_SECONDS

    @torch.no_grad()
    def _evaluate(self, a):
        psnr_sum, ssim_sum, lpips_sum, l1_sum, n_views = 0.0, 0.0, 0.0, 0.0, 0
        eval_h, eval_w = 0, 0
        for view in _iter_eval_views(a):
            image, gt = _render_pair(a, view)
            eval_h, eval_w = int(image.shape[-2]), int(image.shape[-1])
            image_b = image.unsqueeze(0)
            gt_b = gt.unsqueeze(0)
            psnr_sum += _repo_psnr(image_b, gt_b).mean().item()
            ssim_sum += _repo_ssim(image, gt).mean().item()
            l1_sum += _repo_l1(image, gt).mean().item()
            lpips_sum += _lpips_value(image, gt)
            n_views += 1
        n_views = max(n_views, 1)
        return {
            "psnr": psnr_sum / n_views,
            "ssim": ssim_sum / n_views,
            "lpips": lpips_sum / n_views,
            "eval_l1_loss": l1_sum / n_views,
            "num_eval_views": n_views,
            "eval_resolution": [eval_h, eval_w],
        }

    def step(self, a):
        now = time.time()
        if self.t_start is None:
            self.t_start = now
            self.last_eval_time = now

        iteration = int(_global_iteration(a))
        stage = _stage_of(a)

        if not _stage_is_evaluable(stage):
            return
        if not self._must_evaluate(a, iteration, now):
            return

        eval_t0 = time.time()
        torch.cuda.empty_cache()
        metrics = self._evaluate(a)
        torch.cuda.synchronize()
        eval_dt = time.time() - eval_t0

        # Both loss variants are always recorded (SSIM is computed anyway)
        eval_photometric = ((1.0 - LAMBDA_DSSIM) * metrics["eval_l1_loss"]
                            + LAMBDA_DSSIM * (1.0 - metrics["ssim"]))
        eval_loss = (eval_photometric if EVAL_LOSS_KIND == "l1_dssim"
                     else metrics["eval_l1_loss"])

        wall = eval_t0 - self.t_start
        entry = {
            "iteration": iteration,
            "total_iterations": iteration + ITERATION_OFFSET,
            "images_seen": (iteration + ITERATION_OFFSET) * BATCH_SIZE,
            "stage": stage if stage is not None else "single_stage",
            "training_time_s": wall - self.eval_overhead_s,
            "wall_time_s": wall,
            "benchmark_overhead_s": self.eval_overhead_s,
            "psnr": metrics["psnr"],
            "ssim": metrics["ssim"],
            "lpips": metrics["lpips"],
            "eval_l1_loss": metrics["eval_l1_loss"],
            "eval_photometric_loss": eval_photometric,
            "eval_loss": eval_loss,
            "eval_loss_kind": EVAL_LOSS_KIND,
            "num_eval_views": metrics["num_eval_views"],
            "eval_resolution": metrics["eval_resolution"],
            "num_gaussians": int(_num_gaussians(a)),
            "train_batch_loss": _train_batch_loss(a),
            "eval_duration_s": eval_dt,
        }
        entry.update(_extra_entry_fields(a))

        self.last_eval_iteration = iteration
        self.n_evals += 1

        _SUMMARY["entries"].append(entry)
        _SUMMARY["final"] = entry
        _flush_summary()

        print("\n[benchmark] iter %7d | train %8.1fs | PSNR %6.3f | SSIM %.4f | "
              "LPIPS %.4f | eval_loss %.5f | #G %d | eval@%dx%d on %d views"
              % (iteration, entry["training_time_s"], entry["psnr"], entry["ssim"],
                 entry["lpips"], entry["eval_loss"], entry["num_gaussians"],
                 entry["eval_resolution"][1], entry["eval_resolution"][0],
                 entry["num_eval_views"]),
              flush=True)

        _maybe_track_best(a, entry, self)

        # Evaluation, JSON flush and best-checkpoint writes are benchmark overhead, not training time.
        self.eval_overhead_s += time.time() - eval_t0
        self.last_eval_time = time.time()

        if MODE == "target_eval_loss":
            if self._target_met(entry):
                self.consecutive_hits += 1
                if TARGET_METRIC == "psnr":
                    print("[benchmark] target met (%d/%d consecutive): PSNR %.4f >= %.4f"
                          % (self.consecutive_hits, TARGET_CONSECUTIVE_HITS,
                             entry["psnr"], TARGET_PSNR), flush=True)
                else:
                    print("[benchmark] target met (%d/%d consecutive): eval_loss "
                          "%.6f <= %.6f" % (self.consecutive_hits,
                                            TARGET_CONSECUTIVE_HITS,
                                            eval_loss, TARGET_EVAL_LOSS), flush=True)
            else:
                if self.consecutive_hits:
                    print("[benchmark] target lost again, consecutive counter reset.",
                          flush=True)
                self.consecutive_hits = 0

            if self.consecutive_hits >= TARGET_CONSECUTIVE_HITS:
                criterion = ("PSNR >= %.4f" % TARGET_PSNR if TARGET_METRIC == "psnr"
                             else "eval_loss <= %.6f" % TARGET_EVAL_LOSS)
                print("\n[benchmark] TARGET REACHED (%s held for %d consecutive "
                      "sampling points) at iteration %d. Saving the model and "
                      "stopping training." % (criterion, TARGET_CONSECUTIVE_HITS,
                                              iteration), flush=True)
                _SUMMARY["target_reached"] = True
                _SUMMARY["status"] = "stopped_on_target"
                _SUMMARY["target_metric"] = TARGET_METRIC
                _save_model(a)
                _flush_summary()
                sys.stdout.flush()
                sys.exit(0)
            if iteration >= SAFETY_MAX_ITERATIONS:
                print("\n[benchmark] Safety cap of %d iterations reached without "
                      "hitting the target eval loss. Saving and stopping."
                      % SAFETY_MAX_ITERATIONS, flush=True)
                _SUMMARY["status"] = "stopped_on_safety_cap"
                _save_model(a)
                _flush_summary()
                sys.stdout.flush()
                sys.exit(0)


_MONITOR = _Monitor()


# Hook factory, called from the block injected at the bottom of train.py

def make_hook(original_hook):
    signature = inspect.signature(original_hook)

    def hooked(*args, **kwargs):
        result = original_hook(*args, **kwargs)
        try:
            bound = signature.bind(*args, **kwargs)
            bound.apply_defaults()
            arguments = dict(bound.arguments)
            arguments["_caller_locals"] = _caller_locals()
            _MONITOR.step(arguments)
        except SystemExit:
            raise
        except Exception as exc:
            print("[benchmark] evaluation hook failed: %r" % (exc,), flush=True)
        return result

    print("[benchmark] monitor installed (method=%s, scene=%s, mode=%s, frames=%d)"
          % (METHOD, SCENE, MODE, NUM_FRAMES), flush=True)
    return hooked


def _caller_locals(depth=2):
    """Locals of the training loop that called the wrapped function.

    Two of the three repositories pass everything the monitor needs as arguments
    (as `training_report(...)` does); the third calls a per-iteration function
    with a narrower signature, and the loop state has to be read from the calling
    frame instead. The frame is only read, never written.
    """
    # depth 0 = this function, 1 = the wrapper in make_hook, 2 = the training loop.
    frame = sys._getframe(depth)
    return frame.f_locals


def _extra_entry_fields(a):
    """Per-method extra columns; the glue overrides this when it has any."""
    return {}


# === END BENCHMARK MONITOR CORE ===

# Repository-specific glue: oppo-us-research/SpacetimeGaussians.
# controlgaussians(opt, gaussians, densify, iteration, scene, visibility_filter, radii,
#                  viewspace_point_tensor, flag, ...) is called once per iteration inside the
# loop's torch.no_grad() block, and carries the iteration, the model and the scene. The render
# function, the pipeline parameters, the background, the radial-basis function and the
# rasterizer classes are locals of train(), read from the calling frame.

from helper_train import getgtisint8


def _stage_is_evaluable(stage):
    return True


def _stage_of(a):
    return None


def _global_iteration(a):
    return int(a["iteration"])


def _num_gaussians(a):
    return a["gaussians"]._xyz.shape[0]


def _train_batch_loss(a):
    loss = a.get("_caller_locals", {}).get("loss", None)
    return float(loss.item()) if hasattr(loss, "item") else None


def _force_eval(a):
    return False


def _iter_eval_views(a):
    # The test split is the held-out camera over the whole window: readColmapSceneInfo sorts
    # the cameras by name and takes the first `duration` entries, which are its frames.
    cameras = a["scene"].getTestCameras()
    total = len(cameras)
    indices = list(range(total))
    if MAX_EVAL_VIEWS and MAX_EVAL_VIEWS < total:
        step = max(1, total // MAX_EVAL_VIEWS)
        indices = indices[::step][:MAX_EVAL_VIEWS]
    for index in indices:
        yield cameras[index]


def _render_pair(a, view):
    loop = a["_caller_locals"]
    render_pkg = loop["render"](view, a["gaussians"], loop["pipe"], loop["background"],
                                override_color=None,
                                basicfunction=loop["rbfbasefunction"],
                                GRsetting=loop["GRsetting"], GRzer=loop["GRzer"])
    image = torch.clamp(render_pkg["render"], 0.0, 1.0)
    if getgtisint8():
        gt = view.original_image.cuda().float() / 255.0
    else:
        gt = view.original_image.float().cuda()
    return image, torch.clamp(gt, 0.0, 1.0)[0:3, :, :]


def _save_model(a):
    iteration = int(a["iteration"])
    a["scene"].save(iteration)
    print("[benchmark] point cloud saved at iteration %d" % iteration, flush=True)


def _maybe_track_best(a, entry, monitor):
    if entry["psnr"] > monitor.best_psnr:
        monitor.best_psnr = entry["psnr"]
        _SUMMARY["best_psnr"] = entry["psnr"]
        _SUMMARY["best_psnr_iteration"] = entry["iteration"]
'''

with open(os.path.join(REPO_DIR, "benchmark_monitor.py"), "w") as handle:
    handle.write(MONITOR_SRC)

# Inject the hook into train.py (idempotent; a pristine copy is kept as train.py.orig).
# controlgaussians() is defined in another module and imported into train.py's namespace, so
# rebinding the global name in train.py is enough: the loop resolves it at call time.
TRAIN_PY = os.path.join(REPO_DIR, "train.py")
BACKUP_PY = TRAIN_PY + ".orig"
if not os.path.exists(BACKUP_PY):
    shutil.copyfile(TRAIN_PY, BACKUP_PY)
shutil.copyfile(BACKUP_PY, TRAIN_PY)

with open(TRAIN_PY, "r") as handle:
    source = handle.read()

INJECTED = (
    "# === BENCHMARK HOOK (injected by the notebook) ===\n"
    "import os as _bench_os\n"
    "if _bench_os.environ.get('BENCH_CONFIG'):\n"
    "    import benchmark_monitor as _bench\n"
    "    controlgaussians = _bench.make_hook(controlgaussians)\n"
    "# === END BENCHMARK HOOK ===\n\n"
)
ANCHOR = 'if __name__ == "__main__":'
assert source.count(ANCHOR) == 1, "Unexpected train.py layout."
assert ("controlgaussians" in source), "controlgaussians is not referenced in train.py."
with open(TRAIN_PY, "w") as handle:
    handle.write(source.replace(ANCHOR, INJECTED + ANCHOR, 1))

print("benchmark_monitor.py written and the hook injected into train.py (controlgaussians).")
print("Without the BENCH_CONFIG environment variable train.py behaves exactly "
      "as upstream (pristine copy kept as train.py.orig).")


# Peak VRAM: training runs in a subprocess, so nvidia-smi is polled for the whole device.
class VramSampler(threading.Thread):

    def __init__(self, period_s=2.0):
        threading.Thread.__init__(self)
        self.daemon = True
        self.period_s = period_s
        self.peak_mb = 0.0
        self._stop_event = threading.Event()

    def run(self):
        while not self._stop_event.is_set():
            try:
                out = subprocess.check_output(
                    ["nvidia-smi", "--query-gpu=memory.used",
                     "--format=csv,noheader,nounits"])
                self.peak_mb = max(self.peak_mb,
                                   float(out.decode().strip().splitlines()[0]))
            except Exception:
                pass
            self._stop_event.wait(self.period_s)

    def stop(self):
        self._stop_event.set()
        return self.peak_mb


def write_bench_config(p):
    """Serialise the benchmark settings read by benchmark_monitor.py."""
    target = TARGET_EVAL_LOSS_PER_SCENE.get(p["scene"])
    config = {
        "method": METHOD_NAME,
        "scene": p["scene"],
        "mode": p["mode"],
        "json_path": p["bench_json"],
        "eval_every_n_iters": int(EVAL_EVERY_N_ITERS),
        "eval_every_n_seconds": float(EVAL_EVERY_N_MINUTES) * 60.0,
        "max_iterations": int(MAX_ITERATIONS),
        "safety_max_iterations": int(SAFETY_MAX_ITERATIONS),
        "target_metric": TARGET_METRIC,
        "target_psnr": float(TARGET_PSNR),
        "target_eval_loss": float(target) if target is not None else 0.0,
        "min_iterations_before_stop": int(MIN_ITERATIONS_BEFORE_STOP),
        "target_consecutive_hits": int(TARGET_CONSECUTIVE_HITS),
        "eval_loss_kind": EVAL_LOSS_KIND,
        "lambda_dssim": float(LAMBDA_DSSIM),
        "lpips_net": LPIPS_NET,
        "lpips_backend": LPIPS_BACKEND,
        "max_eval_views": int(MAX_EVAL_VIEWS),
        "eval_at_first_iteration": bool(EVAL_AT_FIRST_ITERATION),
        "source_path": p["source_path"],
        "model_path": p["model_path"],
        "batch_size": int(BATCH_SIZE),
        "iteration_offset": int(globals().get("COARSE_ITERATIONS", 0)),
        "dataset": "n3dv",
        # A block of a run trained in blocks covers fewer frames than the study window.
        "num_frames": int(p.get("num_frames", NUM_FRAMES)),
        "held_out_camera": HELD_OUT_CAMERA,
        "image_downscale": int(IMAGE_DOWNSCALE),
    }
    config.update(extra_bench_config(p))
    os.makedirs(p["benchmark_dir"], exist_ok=True)
    with open(p["bench_config"], "w") as handle:
        json.dump(config, handle, indent=2)
    return config


def record_fields(p, **fields):
    """Add run-level fields to the benchmark JSON of one run."""
    if not os.path.exists(p["bench_json"]):
        return
    with open(p["bench_json"], "r") as handle:
        summary = json.load(handle)
    summary.update(fields)
    with open(p["bench_json"], "w") as handle:
        json.dump(summary, handle, indent=2)


def record_run_cost(p, wall_s, peak_vram_mb):
    record_fields(p, process_wall_time_s=wall_s, peak_vram_mb_nvidia_smi=peak_vram_mb)


def scene_data_mb(scene):
    """Disk taken by the prepared data of one scene: the extracted frames and whatever
    the method built from them. Links are counted as links, so frames shared through
    symlinks are not counted twice."""
    roots = [raw_scene_dir(scene), source_path_for(scene)]
    roots += globals().get("scene_data_dirs", lambda s: [])(scene)
    seen, total = set(), 0
    for root in roots:
        for dirpath, _dirs, filenames in os.walk(root):
            for filename in filenames:
                path = os.path.join(dirpath, filename)
                if path in seen:
                    continue
                seen.add(path)
                try:
                    total += os.lstat(path).st_size
                except OSError:
                    pass
    return round(total / 1048576.0, 1)


def run_training_process(p, cmd, budget):
    """Run one training command with the monitor attached; return (returncode, wall_s, peak_mb).

    The loop needs this to be a function, so the training command is issued
    with subprocess rather than the `!` magic (which cannot appear inside a
    function body). Output is streamed line by line, as `!` would do.
    """
    print("=" * 78)
    print("[%s | %s] %s" % (p["scene"], p["mode"], cmd))
    print("=" * 78, flush=True)

    env = dict(os.environ, BENCH_CONFIG=p["bench_config"])
    env.update(extra_train_env(p, budget))
    sampler = VramSampler()
    sampler.start()
    t0 = time.time()
    proc = subprocess.Popen(cmd, shell=True, cwd=REPO_DIR, env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    proc.wait()
    wall = time.time() - t0
    peak = sampler.stop()
    record_run_cost(p, wall, peak)
    return proc.returncode, wall, peak


def record_storage(p):
    """Measure the model storage of one run into its JSON, before the model is discarded."""
    try:
        storage = model_storage_report(p["model_path"])
        with open(p["bench_json"], "r") as handle:
            summary = json.load(handle)
        summary["storage"] = storage
        with open(p["bench_json"], "w") as handle:
            json.dump(summary, handle, indent=2)
        print("[%s] model storage %.2f MB (full folder %.2f MB)"
              % (p["scene"], storage["model_storage_mb"], storage["full_folder_mb"]))
    except Exception as exc:
        print("[%s] storage measurement failed: %r" % (p["scene"], exc))


def train_scene(scene, mode=None):
    """Train one scene under one protocol. Returns a short status string."""
    mode = mode or TRAINING_MODE
    # A method whose official unit is shorter than the study window (Spacetime Gaussians
    # beyond 50 frames) trains the scene as several independent blocks instead.
    if globals().get("NUM_BLOCKS", 1) > 1:
        return train_scene_in_blocks(scene, mode)
    p = activate_scene(scene, mode)

    if mode == "target_eval_loss" and TARGET_METRIC == "eval_loss":
        if TARGET_EVAL_LOSS_PER_SCENE.get(scene) is None:
            print("[%s] no target calibrated for this scene -> skipped." % scene)
            return "skipped_no_target"

    if SKIP_TRAINING_IF_TRAINED and run_is_complete(p):
        print("[%s] already benchmarked -> skipped (%s)."
              % (scene, p["bench_json"]))
        return "skipped_already_trained"

    os.chdir(REPO_DIR)
    _t_prep = time.time()
    extra = prepare_scene(scene, mode)
    prep_s = time.time() - _t_prep
    data_mb = scene_data_mb(scene)
    write_bench_config(p)
    budget = MAX_ITERATIONS if mode == "iterations" else SAFETY_MAX_ITERATIONS
    cmd = build_train_command(p, budget, extra)
    returncode, wall, peak = run_training_process(p, cmd, budget)
    # Preparation (download, frame extraction, COLMAP, conversion) is not training time,
    # but it is GPU-machine time and disk: recorded for the cost estimates. It is the
    # time of this invocation, so near zero when the scene was already prepared.
    record_fields(p, preparation_time_s=prep_s, scene_data_mb=data_mb)

    # Measure model storage now, before the local model is discarded.
    record_storage(p)

    if KEEP_MODEL_ON_DRIVE:
        print("[%s] model kept at %s" % (scene, p["model_path"]))
    else:
        shutil.rmtree(p["model_path"], ignore_errors=True)
        print("[%s] local model discarded; metrics kept on Drive." % scene)

    # The JSON is final at this point: put it on Drive before anything else can go wrong.
    try:
        sync_push(p["run_name"])
    except Exception as exc:
        print("[%s] sync failed: %r (results are still in %s)"
              % (scene, exc, p["benchmark_dir"]))

    if DELETE_FRAMES_AFTER_TRAIN:
        try:
            release_scene(scene)
        except Exception as exc:
            print("[%s] could not free the scene data: %r" % (scene, exc))

    print("\n[%s] finished in %dm %ds (return code %d), peak VRAM %.0f MB"
          % (scene, int(wall // 60), int(wall % 60), returncode, peak),
          flush=True)
    return "ok" if returncode == 0 else "failed(%d)" % returncode


print("Training driver ready.")

### 4.1 Single-scene training

Trains `SCENE` under the protocol selected by `TRAINING_MODE`. Inert unless `RUN_MODE == "single"`.

In [ ]:
# 4.1  Train a single scene (inert unless RUN_MODE == "single")
if RUN_MODE != "single":
    print("RUN_MODE is 'loop': use cell 4.2 instead.")
else:
    status = train_scene(SCENE)
    print("Status: %s" % status)

### 4.2 Training loop

Trains every scene in `SCENES_TO_RUN` sequentially, one run fully written to storage before the next starts. Inert unless `RUN_MODE == "loop"`. The loop is resumable: completed runs are skipped, so re-running the cell after a disconnection or a GPU-quota stop continues where it stopped. A scene that raises an exception is recorded and the loop moves on. Scenes run one at a time on purpose, since sharing the GPU would corrupt the time and VRAM measurements — and, in the multi-view setting, because the extracted frames of a single scene already fill most of the runtime disk.

In [ ]:
# 4.2  Train every scene in SCENES_TO_RUN (re-run the cell to resume after a disconnection)
import time

if RUN_MODE != "loop":
    print("RUN_MODE is 'single': use cell 4.1 instead.")
else:
    loop_results = {}
    loop_t0 = time.time()
    for _i, _scene in enumerate(SCENES_TO_RUN, 1):
        print("\n\n### [%d/%d] %s ###" % (_i, len(SCENES_TO_RUN), _scene),
              flush=True)
        try:
            loop_results[_scene] = train_scene(_scene)
        except KeyboardInterrupt:
            loop_results[_scene] = "interrupted"
            break
        except Exception as _exc:
            loop_results[_scene] = "error: %r" % (_exc,)
            print("[%s] ERROR: %r -- continuing with the next scene."
                  % (_scene, _exc), flush=True)

    print("\n" + "=" * 60)
    print(" LOOP SUMMARY (%s, %.1f min)"
          % (TRAINING_MODE, (time.time() - loop_t0) / 60.0))
    print("=" * 60)
    for _scene in SCENES_TO_RUN:
        print("  %-16s %s" % (_scene, loop_results.get(_scene, "not reached")))

### 4.3 Protocol B target calibration

Run after the Protocol A loop. Prints this method's final eval L1 per scene and the candidate target with a 5% margin. The target used in `TARGET_EVAL_LOSS_PER_SCENE` is the **largest candidate across the three methods**, identical in all three notebooks; the margin keeps the slowest method from meeting the target only where its curve flattens.

Note: as in the monocular study, the targets should apply the same rule to the **minimum** of each Protocol A test curve rather than to the final sample, because a method whose test curve peaks and then degrades would otherwise set the bar at a worse model than it can reach. The cell prints both.

In [ ]:
# 4.3  Protocol B calibration: prints this method's candidate targets only. Take the per-scene
# maximum over the three methods and paste it into all three notebooks.
import json
import os

print("%-18s %14s %14s %14s" % ("scene", "final eval L1", "min eval L1", "candidate x1.05"))
print("-" * 64)
candidates = {}
for _scene in N3DV_SCENES:
    _p = paths_for(_scene, "iterations")
    if not os.path.exists(_p["bench_json"]):
        print("%-18s %14s %14s %14s" % (_scene, "-", "-", "-"))
        continue
    with open(_p["bench_json"], "r") as handle:
        _summary = json.load(handle)
    _entries = _summary.get("entries") or []
    _final = _summary.get("final")
    if not _final or not _entries:
        print("%-18s %14s %14s %14s" % (_scene, "-", "-", "-"))
        continue
    _l1 = _final["eval_l1_loss"]
    _l1_min = min(e["eval_l1_loss"] for e in _entries)
    candidates[_scene] = round(_l1_min * 1.05, 6)
    print("%-18s %14.6f %14.6f %14.6f" % (_scene, _l1, _l1_min, candidates[_scene]))

print("\nThis method's candidates (compare with the other two, keep the max):")
print(json.dumps(candidates, indent=4))

### 4.4 Benchmark summary

Reads the JSON of the active scene. After a loop, call `activate_scene("<scene>")` to inspect another one.

In [ ]:
# 4.4  Benchmark summary for the active scene
import json
import os

import pandas as pd

train_duration = None
benchmark_summary = None
benchmark_table = None

if os.path.exists(BENCH_JSON_PATH):
    with open(BENCH_JSON_PATH, "r") as handle:
        benchmark_summary = json.load(handle)
    entries = benchmark_summary.get("entries", [])
else:
    entries = []

if entries:
    _all = pd.DataFrame(entries)
    columns = [c for c in ["iteration", "total_iterations", "images_seen",
                           "training_time_s", "psnr", "ssim", "lpips",
                           "eval_loss", "eval_l1_loss", "eval_photometric_loss",
                           "num_gaussians", "stage"] if c in _all.columns]
    benchmark_table = _all[columns]
    pd.set_option("display.float_format", lambda v: "%.5f" % v)

    print("Method   : %s" % benchmark_summary["method"])
    print("Scene    : %s" % benchmark_summary["scene"])
    print("Protocol : %s" % benchmark_summary["mode"])
    print("Status   : %s" % benchmark_summary["status"])
    print("Window   : %s frames, held-out camera %s"
          % (benchmark_summary.get("num_frames"),
             benchmark_summary.get("held_out_camera")))

    final = benchmark_summary["final"]
    train_duration = final["training_time_s"]

    if benchmark_summary["mode"] == "target_eval_loss":
        _cfg = benchmark_summary.get("config", {})
        _target = _cfg.get("target_eval_loss")
        print("Target   : eval_loss <= %.6f, held for %d consecutive samples"
              % (_target, _cfg.get("target_consecutive_hits", 2)))
        print("Reached  : %s" % benchmark_summary.get("target_reached"))
        # Report the first crossing of the target, not the termination (which includes the hysteresis).
        _first = next((e for e in entries if e["eval_loss"] <= _target), None)
        if _first is not None:
            print("\n  First crossing  : iteration %d, %.1f s (%.2f min)"
                  % (_first["iteration"], _first["training_time_s"],
                     _first["training_time_s"] / 60.0))
            print("  Stop confirmed  : iteration %d, %.1f s (%.2f min)"
                  % (final["iteration"], train_duration, train_duration / 60.0))
            print("  Hysteresis delay: %.1f s (+%.0f%%)"
                  % (train_duration - _first["training_time_s"],
                     100.0 * (train_duration / max(_first["training_time_s"], 1e-9) - 1.0)))
            print("  --> report the FIRST CROSSING as the cost of reaching the target.")
    print()
    display(benchmark_table)

    print()
    print("=" * 64)
    print(" FINAL POINT OF THE RUN")
    print("=" * 64)
    print("  Iterations                : %d" % final["iteration"])
    print("  Training time (net)       : %.2f s (%.2f min)"
          % (train_duration, train_duration / 60.0))
    print("  Benchmark overhead        : %.2f s" % final["benchmark_overhead_s"])
    print("  PSNR                      : %.4f dB" % final["psnr"])
    print("  SSIM                      : %.4f" % final["ssim"])
    print("  LPIPS                     : %.4f" % final["lpips"])
    print("  Eval loss (%-8s)      : %.6f" % (final["eval_loss_kind"], final["eval_loss"]))
    print("  Number of Gaussians       : %d" % final["num_gaussians"])
    print("  Test views evaluated      : %d  <-- must be %d across methods"
          % (final.get("num_eval_views", -1), NUM_FRAMES))
    if "eval_resolution" in final:
        print("  Evaluation resolution     : %d x %d  <-- must match across methods"
              % (final["eval_resolution"][1], final["eval_resolution"][0]))
    if "peak_vram_mb_nvidia_smi" in benchmark_summary:
        print("  Peak VRAM (device)        : %.0f MB"
              % benchmark_summary["peak_vram_mb_nvidia_smi"])
    print("=" * 64)
    print("JSON: %s" % BENCH_JSON_PATH)
else:
    print("No benchmark data for the active scene (%s, %s)." % (SCENE, TRAINING_MODE))

if benchmark_table is not None and len(benchmark_table) > 1:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    minutes = benchmark_table["training_time_s"] / 60.0
    axes[0].plot(minutes, benchmark_table["psnr"], marker="o", color="tab:blue")
    axes[0].set_xlabel("training time [min]"); axes[0].set_ylabel("PSNR [dB]")
    axes[0].set_title("PSNR (higher is better)")
    axes[1].plot(minutes, benchmark_table["ssim"], marker="o", color="tab:green")
    axes[1].set_xlabel("training time [min]"); axes[1].set_ylabel("SSIM")
    axes[1].set_title("SSIM (higher is better)")
    axes[2].plot(minutes, benchmark_table["lpips"], marker="s", color="tab:red")
    axes[2].set_xlabel("training time [min]"); axes[2].set_ylabel("LPIPS")
    axes[2].set_title("LPIPS (lower is better)")
    axes[3].plot(benchmark_table["iteration"], benchmark_table["eval_loss"],
                 marker="o", color="tab:orange")
    axes[3].set_xlabel("iteration"); axes[3].set_ylabel("evaluation loss")
    axes[3].set_title("Evaluation loss (lower is better)")
    for ax in axes:
        ax.grid(alpha=0.3)
    fig.suptitle("%s -- %s (%s, %d frames)"
                 % (METHOD_NAME, SCENE, TRAINING_MODE, NUM_FRAMES))
    plt.tight_layout()
    plt.show()

---

# Part 2: evaluation and visualisation

**Do not run this part while a training loop is in progress.** Run cells 0.1 and 0.2 first (and the environment cells after a runtime restart), then call `activate_scene("<scene>")` to select the trained model. With `KEEP_MODEL_ON_DRIVE = False` the model exists only in the runtime that trained it, and with `DELETE_FRAMES_AFTER_TRAIN = True` the ground-truth frames were discarded too — re-run cell 3 for the scene before rendering.

The real-time WebSocket viewer of the monocular notebooks is not reproduced here: it adds nothing to the measurements and an N3DV model is too large to stream from a cloud runtime. Rendering, final metrics and an MP4 of the held-out view are kept.

---

## 5. Rendering and final metrics

`test.py` renders the held-out camera over the window with the forward-only rasterizer and writes the metrics. It needs the `forward_lite` (or `forward_full`) module, which the setup cell did not build because training does not use it, so it is installed here.

In [ ]:
# The forward-only rasterizer is needed for rendering, not for training.
import os

_module = "forward_lite" if STG_MODEL == "ours_lite" else "forward_full"
os.chdir(os.path.join(REPO_DIR, "thirdparty/gaussian_splatting/submodules", _module))
!pip install . --no-build-isolation
os.chdir(REPO_DIR)
print("%s installed." % _module)

In [ ]:
import os

# A run trained in blocks (more than 50 frames) is NUM_BLOCKS separate models: pick the
# block to render. Its benchmark metrics are already in the merged JSON of section 4.
RENDER_BLOCK = 0
if NUM_BLOCKS > 1:
    _q = block_paths(paths_for(SCENE, TRAINING_MODE), RENDER_BLOCK)
    RENDER_SOURCE, RENDER_DIR, RENDER_FRAMES = _q["source_path"], _q["model_path"], STG_BLOCK_FRAMES
    print("Rendering block %d of %d (frames %d-%d)" % (RENDER_BLOCK, NUM_BLOCKS,
          _q["block_start"], _q["block_start"] + STG_BLOCK_FRAMES - 1))
else:
    RENDER_SOURCE, RENDER_DIR, RENDER_FRAMES = SOURCE_PATH, MODEL_DIR, NUM_FRAMES

os.chdir(REPO_DIR)
_cfg = config_path(SCENE)
!python test.py --quiet --eval --skip_train \
    --configpath "{_cfg}" \
    --source_path "{RENDER_SOURCE}" \
    --model_path "{RENDER_DIR}" \
    --model {STG_MODEL} --rdpip test_{STG_MODEL} \
    --duration {RENDER_FRAMES} --resolution 1 --valloader colmapvalid

In [ ]:
import glob
import json
import os

for path in sorted(glob.glob(os.path.join(RENDER_DIR, "*.json"))):
    print("--- %s" % os.path.basename(path))
    with open(path) as handle:
        print(json.dumps(json.load(handle), indent=2))
print()
print("These come from 8-bit PNGs written by test.py, while the monitor evaluates float")
print("tensors during training: small differences between the two are expected.")

## 6. Video export of the held-out view

Assembles the rendered frames of the held-out camera into an MP4, next to the ground truth.

In [ ]:
# Encode the rendered test frames into a side-by-side MP4 (render | ground truth)
import glob
import os

FPS_OUT = 30
renders = sorted(glob.glob(os.path.join(RENDER_DIR, "test", "ours_*", "renders", "*.png")))
truths = sorted(glob.glob(os.path.join(RENDER_DIR, "test", "ours_*", "gt", "*.png")))
print("%d rendered frames, %d ground-truth frames" % (len(renders), len(truths)))

if renders and truths:
    out = os.path.join(RENDER_DIR, "%s_test.mp4" % RUN_NAME)
    render_glob = os.path.join(os.path.dirname(renders[0]), "%05d.png")
    gt_glob = os.path.join(os.path.dirname(truths[0]), "%05d.png")
    !ffmpeg -y -loglevel error -framerate {FPS_OUT} -i "{render_glob}" -framerate {FPS_OUT} -i "{gt_glob}" \
        -filter_complex hstack -c:v libx264 -pix_fmt yuv420p -crf 18 "{out}"
    print("written: %s" % out)

    from IPython.display import HTML
    import base64
    with open(out, "rb") as handle:
        payload = base64.b64encode(handle.read()).decode()
    display(HTML('<video width=960 controls><source src="data:video/mp4;base64,%s"></video>'
                 % payload))
else:
    print("Nothing to encode: run section 5 first.")